In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv


In [5]:
%%writefile _common.py
import re
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd


CANONICAL_COLUMNS = [
    "name",
    "country_origin",
    "domain",
    "price_category",
    "founded",
    "presence_world",
    "presence_russia",
    "presence_regions",
    "description",
    "plans",
    "total_rented_area",
]


def harmonize_schema(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "country_origin" not in df.columns and "contry_origin" in df.columns:
        df = df.rename(columns={"contry_origin": "country_origin"})
    if "country_origin" in df.columns and "contry_origin" in df.columns:
        df = df.drop(columns=["contry_origin"])
    return df


def normalize_basic_text(s: object) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = s.replace("\u200b", "").replace("\xa0", " ")
    s = re.sub(r"\s+", " ", s)
    return s


def normalize_name_for_audit(s: object) -> str:
    s = normalize_basic_text(s)
    s = re.sub(r"[^a-zа-я0-9 ]", " ", s)
    s = re.sub(r"\b(?:ооо|зао|ип|оао|ooo|zao)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    words = []
    seen = set()
    for w in s.split():
        if w not in seen:
            words.append(w)
            seen.add(w)
    return " ".join(words)


def normalize_country_for_audit(s: object) -> str:
    s = normalize_basic_text(s)
    s = re.sub(r"[^\w\s,-]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def normalize_domain_for_audit(s: object) -> str:
    s = normalize_basic_text(s)
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def normalize_price_for_audit(s: object) -> str:
    if pd.isna(s):
        return "неизвестно"
    s = str(s).strip().lower()
    s = s.replace("\u200b", "").replace("\xa0", " ")
    s = s.replace(",", ";").replace(":", ";")
    parts = [p.strip() for p in s.split(";") if p.strip()]
    parts = list(dict.fromkeys(parts))
    return ";".join(parts) if parts else "неизвестно"


def parse_presence_russia(text: object) -> Tuple[float, float]:
    if pd.isna(text):
        return np.nan, np.nan

    text = str(text)
    nums = list(map(int, re.findall(r"\d+", text)))
    low = text.lower()

    if "франчайз" in low:
        if len(nums) >= 2:
            return float(nums[0]), float(nums[1])
        if len(nums) == 1:
            return 0.0, float(nums[0])

    if len(nums) >= 1:
        return float(nums[0]), 0.0

    return np.nan, np.nan


def count_regions(text: object) -> int:
    if pd.isna(text):
        return 0
    return len([x.strip() for x in str(text).split(";") if x.strip()])


def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")

Overwriting _common.py


In [6]:
%%writefile audit_schema_and_missing.py
import argparse
import json
from pathlib import Path

import pandas as pd

from _common import CANONICAL_COLUMNS, harmonize_schema, save_csv


def audit_schema(df: pd.DataFrame) -> dict:
    return {
        "n_rows": int(df.shape[0]),
        "n_cols": int(df.shape[1]),
        "columns": list(df.columns),
        "missing_canonical_columns": [c for c in CANONICAL_COLUMNS if c not in df.columns],
    }


def audit_missing(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    n = len(df)

    for col in df.columns:
        missing = int(df[col].isna().sum())
        rows.append({
            "column": col,
            "missing_count": missing,
            "missing_rate": float(missing / n) if n else 0.0,
            "dtype": str(df[col].dtype),
        })

    return pd.DataFrame(rows).sort_values(
        by=["missing_rate", "missing_count"],
        ascending=[False, False]
    )


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--csv_path", type=str, required=True)
    parser.add_argument("--out_dir", type=str, required=True)
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df_raw = pd.read_csv(args.csv_path)
    df = harmonize_schema(df_raw)

    schema_info = audit_schema(df)
    missing_report = audit_missing(df)

    (out_dir / "schema_summary.json").write_text(
        json.dumps(schema_info, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    save_csv(missing_report, out_dir / "missing_report.csv")

    print("audit_schema_and_missing done")


if __name__ == "__main__":
    main()

Writing audit_schema_and_missing.py


In [7]:
%%writefile audit_name.py
import argparse
from pathlib import Path

import pandas as pd

from _common import harmonize_schema, normalize_name_for_audit, save_csv


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--out_dir",
        type=str,
        default="/kaggle/working/preprocessing_audit",
    )
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df = harmonize_schema(pd.read_csv(args.csv_path))

    if "name" not in df.columns:
        raise ValueError("Column 'name' not found")

    temp = df[["name"]].copy()
    temp["name_clean_candidate"] = temp["name"].apply(normalize_name_for_audit)

    raw_counts = (
        temp["name"]
        .fillna("MISSING")
        .astype(str)
        .value_counts()
        .rename_axis("name")
        .reset_index(name="count")
    )

    clean_counts = (
        temp["name_clean_candidate"]
        .fillna("MISSING")
        .astype(str)
        .value_counts()
        .rename_axis("name_clean_candidate")
        .reset_index(name="count")
    )

    ambiguous = (
        temp.groupby("name_clean_candidate")["name"]
        .nunique()
        .reset_index(name="n_original_variants")
        .sort_values(by="n_original_variants", ascending=False)
    )
    ambiguous = ambiguous[ambiguous["n_original_variants"] > 1].copy()

    examples = (
        temp.groupby("name_clean_candidate")["name"]
        .agg(lambda x: " | ".join(sorted(map(str, pd.Series(x).dropna().unique()))))
        .reset_index(name="original_examples")
    )
    examples = examples.merge(
        ambiguous,
        on="name_clean_candidate",
        how="inner"
    ).sort_values(by="n_original_variants", ascending=False)

    save_csv(raw_counts, out_dir / "name_raw_value_counts.csv")
    save_csv(clean_counts, out_dir / "name_clean_candidate_counts.csv")
    save_csv(ambiguous, out_dir / "name_clean_ambiguous_groups.csv")
    save_csv(examples, out_dir / "name_clean_ambiguous_examples.csv")

    print("audit_name done")


if __name__ == "__main__":
    main()

Writing audit_name.py


In [8]:
%%writefile audit_country.py
import argparse
from pathlib import Path

import pandas as pd

from _common import harmonize_schema, normalize_country_for_audit, save_csv


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--out_dir",
        type=str,
        default="/kaggle/working/preprocessing_audit",
    )
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df = harmonize_schema(pd.read_csv(args.csv_path))

    if "country_origin" not in df.columns:
        raise ValueError("Column 'country_origin' not found")

    temp = df[["country_origin"]].copy()
    temp["country_origin_raw"] = temp["country_origin"].fillna("MISSING").astype(str)
    temp["country_origin_norm"] = temp["country_origin"].apply(normalize_country_for_audit)

    raw_counts = (
        temp["country_origin_raw"]
        .value_counts()
        .rename_axis("country_origin_raw")
        .reset_index(name="count")
    )

    norm_counts = (
        temp["country_origin_norm"]
        .value_counts()
        .rename_axis("country_origin_norm")
        .reset_index(name="count")
    )

    raw_to_norm = (
        temp.groupby(["country_origin_raw", "country_origin_norm"])
        .size()
        .reset_index(name="count")
        .sort_values(by=["country_origin_norm", "count"], ascending=[True, False])
    )

    collisions = (
        raw_to_norm.groupby("country_origin_norm")["country_origin_raw"]
        .nunique()
        .reset_index(name="n_raw_variants")
        .sort_values(by="n_raw_variants", ascending=False)
    )
    collisions = collisions[collisions["n_raw_variants"] > 1].copy()

    save_csv(raw_counts, out_dir / "country_raw_value_counts.csv")
    save_csv(norm_counts, out_dir / "country_normalized_value_counts.csv")
    save_csv(raw_to_norm, out_dir / "country_raw_to_normalized_mapping_candidates.csv")
    save_csv(collisions, out_dir / "country_normalized_collision_groups.csv")

    print("audit_country done")


if __name__ == "__main__":
    main()

Writing audit_country.py


In [10]:
%%writefile audit_domain.py
import argparse
from pathlib import Path

import pandas as pd

from _common import harmonize_schema, normalize_domain_for_audit, save_csv


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--out_dir",
        type=str,
        default="/kaggle/working/preprocessing_audit",
    )
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df = harmonize_schema(pd.read_csv(args.csv_path))

    if "domain" not in df.columns:
        raise ValueError("Column 'domain' not found")

    temp = df[["domain"]].copy()
    temp["domain_raw"] = temp["domain"].fillna("MISSING").astype(str)
    temp["domain_norm"] = temp["domain"].apply(normalize_domain_for_audit)

    raw_counts = (
        temp["domain_raw"]
        .value_counts()
        .rename_axis("domain_raw")
        .reset_index(name="count")
    )

    norm_counts = (
        temp["domain_norm"]
        .value_counts()
        .rename_axis("domain_norm")
        .reset_index(name="count")
    )

    raw_to_norm = (
        temp.groupby(["domain_raw", "domain_norm"])
        .size()
        .reset_index(name="count")
        .sort_values(by=["domain_norm", "count"], ascending=[True, False])
    )

    save_csv(raw_counts, out_dir / "domain_raw_value_counts.csv")
    save_csv(norm_counts, out_dir / "domain_normalized_value_counts.csv")
    save_csv(raw_to_norm, out_dir / "domain_raw_to_normalized_mapping_candidates.csv")

    print("audit_domain done")


if __name__ == "__main__":
    main()

Overwriting audit_domain.py


In [11]:
%%writefile audit_price_category.py
import argparse
from pathlib import Path

import pandas as pd

from _common import harmonize_schema, normalize_price_for_audit, save_csv


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--out_dir",
        type=str,
        default="/kaggle/working/preprocessing_audit",
    )
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df = harmonize_schema(pd.read_csv(args.csv_path))

    if "price_category" not in df.columns:
        raise ValueError("Column 'price_category' not found")

    temp = df[["price_category"]].copy()
    temp["price_category_raw"] = temp["price_category"].fillna("MISSING").astype(str)
    temp["price_category_norm"] = temp["price_category"].apply(normalize_price_for_audit)

    raw_counts = (
        temp["price_category_raw"]
        .value_counts()
        .rename_axis("price_category_raw")
        .reset_index(name="count")
    )

    norm_counts = (
        temp["price_category_norm"]
        .value_counts()
        .rename_axis("price_category_norm")
        .reset_index(name="count")
    )

    label_rows = []
    for value in temp["price_category_norm"]:
        parts = [p.strip() for p in str(value).split(";") if p.strip()]
        for p in parts:
            label_rows.append(p)

    label_dist = (
        pd.Series(label_rows)
        .value_counts()
        .rename_axis("price_label")
        .reset_index(name="count")
        if label_rows else pd.DataFrame(columns=["price_label", "count"])
    )

    save_csv(raw_counts, out_dir / "price_category_raw_value_counts.csv")
    save_csv(norm_counts, out_dir / "price_category_normalized_value_counts.csv")
    save_csv(label_dist, out_dir / "price_category_label_distribution.csv")

    print("audit_price_category done")


if __name__ == "__main__":
    main()

Writing audit_price_category.py


In [12]:
%%writefile audit_founded.py
import argparse
from pathlib import Path

import pandas as pd

from _common import harmonize_schema, save_csv


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--out_dir",
        type=str,
        default="/kaggle/working/preprocessing_audit",
    )
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df = harmonize_schema(pd.read_csv(args.csv_path))

    if "founded" not in df.columns:
        raise ValueError("Column 'founded' not found")

    temp = df.copy()
    temp["founded_num"] = pd.to_numeric(temp["founded"], errors="coerce")

    summary = pd.DataFrame([{
        "rows_total": int(len(temp)),
        "rows_non_null": int(temp["founded_num"].notna().sum()),
        "rows_lt_1850": int((temp["founded_num"] < 1850).sum()),
        "rows_gt_2025": int((temp["founded_num"] > 2025).sum()),
        "min": float(temp["founded_num"].min()) if temp["founded_num"].notna().any() else None,
        "median": float(temp["founded_num"].median()) if temp["founded_num"].notna().any() else None,
        "max": float(temp["founded_num"].max()) if temp["founded_num"].notna().any() else None,
    }])

    invalid_rows = temp[
        (temp["founded_num"] < 1850) | (temp["founded_num"] > 2025)
    ].copy()
    keep_cols = [c for c in ["name", "country_origin", "domain", "price_category", "founded"] if c in invalid_rows.columns]
    invalid_rows = invalid_rows[keep_cols].sort_values(by="founded")

    conflicts = pd.DataFrame()
    if all(c in temp.columns for c in ["name", "country_origin", "domain", "price_category"]):
        conflicts = (
            temp.groupby(["name", "country_origin", "domain", "price_category"])["founded_num"]
            .nunique(dropna=True)
            .reset_index(name="n_founded_variants")
            .sort_values(by="n_founded_variants", ascending=False)
        )
        conflicts = conflicts[conflicts["n_founded_variants"] > 1].copy()

    save_csv(summary, out_dir / "founded_summary.csv")
    save_csv(invalid_rows, out_dir / "founded_out_of_range_rows.csv")
    save_csv(conflicts, out_dir / "founded_conflicts.csv")

    print("audit_founded done")


if __name__ == "__main__":
    main()

Writing audit_founded.py


In [13]:
%%writefile audit_presence.py
import argparse
from pathlib import Path

import pandas as pd

from _common import (
    harmonize_schema,
    parse_presence_russia,
    count_regions,
    save_csv,
)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--out_dir",
        type=str,
        default="/kaggle/working/preprocessing_audit",
    )
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df = harmonize_schema(pd.read_csv(args.csv_path))

    if "presence_russia" in df.columns:
        temp = df[["presence_russia"]].copy()
        parsed = temp["presence_russia"].apply(parse_presence_russia)
        temp["presence_own_candidate"] = parsed.apply(lambda x: x[0])
        temp["presence_franchise_candidate"] = parsed.apply(lambda x: x[1])

        summary = pd.DataFrame([{
            "rows_total": int(len(temp)),
            "rows_non_null_raw": int(temp["presence_russia"].notna().sum()),
            "presence_own_non_null": int(temp["presence_own_candidate"].notna().sum()),
            "presence_franchise_non_null": int(temp["presence_franchise_candidate"].notna().sum()),
        }])

        save_csv(summary, out_dir / "presence_russia_summary.csv")
        save_csv(temp.head(200), out_dir / "presence_russia_parsed_preview.csv")

    if "presence_regions" in df.columns:
        temp = df[["presence_regions"]].copy()
        temp["n_regions_candidate"] = temp["presence_regions"].apply(count_regions)

        summary = pd.DataFrame([{
            "rows_total": int(len(temp)),
            "rows_non_null_raw": int(temp["presence_regions"].notna().sum()),
            "mean_n_regions": float(temp["n_regions_candidate"].mean()),
            "median_n_regions": float(temp["n_regions_candidate"].median()),
            "max_n_regions": int(temp["n_regions_candidate"].max()),
        }])

        save_csv(summary, out_dir / "presence_regions_summary.csv")
        save_csv(temp.head(200), out_dir / "presence_regions_preview.csv")

    if "presence_world" in df.columns:
        s = pd.to_numeric(df["presence_world"], errors="coerce")
        summary = pd.DataFrame([{
            "rows_total": int(len(s)),
            "rows_non_null": int(s.notna().sum()),
            "negative_count": int((s < 0).sum()),
            "min": float(s.min()) if s.notna().any() else None,
            "median": float(s.median()) if s.notna().any() else None,
            "max": float(s.max()) if s.notna().any() else None,
        }])
        save_csv(summary, out_dir / "presence_world_summary.csv")

    if "plans" in df.columns:
        s = pd.to_numeric(df["plans"], errors="coerce")
        summary = pd.DataFrame([{
            "rows_total": int(len(s)),
            "rows_non_null": int(s.notna().sum()),
            "negative_count": int((s < 0).sum()),
            "min": float(s.min()) if s.notna().any() else None,
            "median": float(s.median()) if s.notna().any() else None,
            "max": float(s.max()) if s.notna().any() else None,
        }])
        save_csv(summary, out_dir / "plans_summary.csv")

    print("audit_presence done")


if __name__ == "__main__":
    main()

Writing audit_presence.py


In [14]:
%%writefile audit_description.py
import argparse
from pathlib import Path

import pandas as pd

from _common import harmonize_schema, save_csv


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--out_dir",
        type=str,
        default="/kaggle/working/preprocessing_audit",
    )
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df = harmonize_schema(pd.read_csv(args.csv_path))

    if "description" not in df.columns:
        raise ValueError("Column 'description' not found")

    temp = df[["description"]].copy()
    temp["description_text"] = temp["description"].fillna("").astype(str).str.strip()

    temp["desc_len"] = temp["description_text"].str.len()
    temp["desc_word_count"] = temp["description_text"].str.split().apply(len)
    temp["desc_has_digits"] = temp["description_text"].str.contains(r"\d", regex=True).astype(int)
    temp["desc_year_mentions"] = temp["description_text"].str.count(r"\b(?:18|19|20)\d{2}\b")
    temp["desc_is_empty"] = (temp["description_text"] == "").astype(int)

    summary = pd.DataFrame([{
        "rows_total": int(len(temp)),
        "rows_empty": int(temp["desc_is_empty"].sum()),
        "empty_rate": float(temp["desc_is_empty"].mean()),
        "mean_len": float(temp["desc_len"].mean()),
        "median_len": float(temp["desc_len"].median()),
        "max_len": int(temp["desc_len"].max()),
        "mean_word_count": float(temp["desc_word_count"].mean()),
        "median_word_count": float(temp["desc_word_count"].median()),
        "rows_with_digits": int(temp["desc_has_digits"].sum()),
        "rows_with_year_mentions": int((temp["desc_year_mentions"] > 0).sum()),
    }])

    length_distribution = pd.DataFrame([
        {
            "bucket": "0",
            "count": int((temp["desc_len"] == 0).sum()),
        },
        {
            "bucket": "1_50",
            "count": int(((temp["desc_len"] >= 1) & (temp["desc_len"] <= 50)).sum()),
        },
        {
            "bucket": "51_150",
            "count": int(((temp["desc_len"] >= 51) & (temp["desc_len"] <= 150)).sum()),
        },
        {
            "bucket": "151_300",
            "count": int(((temp["desc_len"] >= 151) & (temp["desc_len"] <= 300)).sum()),
        },
        {
            "bucket": "301_600",
            "count": int(((temp["desc_len"] >= 301) & (temp["desc_len"] <= 600)).sum()),
        },
        {
            "bucket": "600_plus",
            "count": int((temp["desc_len"] > 600).sum()),
        },
    ])

    shortest_nonempty = (
        temp[temp["desc_len"] > 0]
        .sort_values(by="desc_len", ascending=True)
        .head(50)
        .copy()
    )

    longest_examples = (
        temp.sort_values(by="desc_len", ascending=False)
        .head(50)
        .copy()
    )

    year_mentions_distribution = (
        temp["desc_year_mentions"]
        .value_counts()
        .sort_index()
        .rename_axis("year_mentions_count")
        .reset_index(name="rows")
    )

    preview = temp[[
        "description_text",
        "desc_len",
        "desc_word_count",
        "desc_has_digits",
        "desc_year_mentions",
        "desc_is_empty",
    ]].head(200)

    save_csv(summary, out_dir / "description_summary.csv")
    save_csv(length_distribution, out_dir / "description_length_distribution.csv")
    save_csv(shortest_nonempty, out_dir / "description_shortest_examples.csv")
    save_csv(longest_examples, out_dir / "description_longest_examples.csv")
    save_csv(year_mentions_distribution, out_dir / "description_year_mentions_distribution.csv")
    save_csv(preview, out_dir / "description_preview.csv")

    print("audit_description done")


if __name__ == "__main__":
    main()

Writing audit_description.py


In [15]:
%%writefile audit_total_rented_area.py
import argparse
from pathlib import Path

import pandas as pd

from _common import harmonize_schema, save_csv


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--out_dir",
        type=str,
        default="/kaggle/working/preprocessing_audit",
    )
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df = harmonize_schema(pd.read_csv(args.csv_path))

    if "total_rented_area" not in df.columns:
        raise ValueError("Column 'total_rented_area' not found")

    temp = df.copy()
    temp["total_rented_area_num"] = pd.to_numeric(temp["total_rented_area"], errors="coerce")
    temp["is_known_total_rented_area"] = temp["total_rented_area_num"].notna().astype(int)

    known = temp[temp["total_rented_area_num"].notna()].copy()

    summary = pd.DataFrame([{
        "rows_total": int(len(temp)),
        "rows_known": int(len(known)),
        "known_rate": float(len(known) / len(temp)) if len(temp) else 0.0,
        "rows_missing": int(temp["total_rented_area_num"].isna().sum()),
        "rows_zero": int((temp["total_rented_area_num"] == 0).sum()),
        "rows_negative": int((temp["total_rented_area_num"] < 0).sum()),
        "min": float(known["total_rented_area_num"].min()) if len(known) else None,
        "p25": float(known["total_rented_area_num"].quantile(0.25)) if len(known) else None,
        "median": float(known["total_rented_area_num"].median()) if len(known) else None,
        "p75": float(known["total_rented_area_num"].quantile(0.75)) if len(known) else None,
        "max": float(known["total_rented_area_num"].max()) if len(known) else None,
        "mean": float(known["total_rented_area_num"].mean()) if len(known) else None,
    }])

    distribution_buckets = pd.DataFrame([
        {
            "bucket": "0_1000",
            "count": int(((known["total_rented_area_num"] >= 0) & (known["total_rented_area_num"] <= 1000)).sum()),
        },
        {
            "bucket": "1001_5000",
            "count": int(((known["total_rented_area_num"] >= 1001) & (known["total_rented_area_num"] <= 5000)).sum()),
        },
        {
            "bucket": "5001_10000",
            "count": int(((known["total_rented_area_num"] >= 5001) & (known["total_rented_area_num"] <= 10000)).sum()),
        },
        {
            "bucket": "10001_50000",
            "count": int(((known["total_rented_area_num"] >= 10001) & (known["total_rented_area_num"] <= 50000)).sum()),
        },
        {
            "bucket": "50001_100000",
            "count": int(((known["total_rented_area_num"] >= 50001) & (known["total_rented_area_num"] <= 100000)).sum()),
        },
        {
            "bucket": "100000_plus",
            "count": int((known["total_rented_area_num"] > 100000).sum()),
        },
    ])

    top_known_examples = pd.DataFrame()
    if len(known):
        keep_cols = [
            c for c in [
                "name",
                "country_origin",
                "domain",
                "price_category",
                "founded",
                "presence_world",
                "presence_russia",
                "presence_regions",
                "plans",
                "total_rented_area_num",
            ]
            if c in known.columns
        ]

        top_known_examples = (
            known[keep_cols]
            .sort_values(by="total_rented_area_num", ascending=False)
            .head(50)
            .copy()
        )

    missing_vs_known_by_domain = pd.DataFrame()
    if "domain" in temp.columns:
        tmp = temp.copy()
        tmp["domain"] = tmp["domain"].fillna("MISSING").astype(str)

        agg = (
            tmp.groupby("domain")["is_known_total_rented_area"]
            .agg(["count", "sum"])
            .reset_index()
            .rename(columns={"count": "rows_total", "sum": "rows_known"})
        )
        agg["known_rate"] = agg["rows_known"] / agg["rows_total"]
        missing_vs_known_by_domain = agg.sort_values(by="known_rate", ascending=False)

    preview = temp[[
        c for c in [
            "name",
            "domain",
            "price_category",
            "presence_world",
            "presence_russia",
            "plans",
            "total_rented_area_num",
            "is_known_total_rented_area",
        ] if c in temp.columns
    ]].head(200)

    save_csv(summary, out_dir / "total_rented_area_summary.csv")
    save_csv(distribution_buckets, out_dir / "total_rented_area_distribution_buckets.csv")
    save_csv(top_known_examples, out_dir / "total_rented_area_top_known_examples.csv")
    save_csv(missing_vs_known_by_domain, out_dir / "total_rented_area_known_rate_by_domain.csv")
    save_csv(preview, out_dir / "total_rented_area_preview.csv")

    print("audit_total_rented_area done")


if __name__ == "__main__":
    main()

Writing audit_total_rented_area.py


In [16]:
%%writefile run_all_audits.py
import argparse
import subprocess
import sys
from pathlib import Path


AUDIT_SCRIPTS = [
    "audit_schema_and_missing.py",
    "audit_name.py",
    "audit_country.py",
    "audit_domain.py",
    "audit_price_category.py",
    "audit_founded.py",
    "audit_presence.py",
    "audit_description.py",
    "audit_total_rented_area.py",
]


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--out_dir",
        type=str,
        default="/kaggle/working/preprocessing_audit",
    )
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    base_dir = Path(".").resolve()

    for script_name in AUDIT_SCRIPTS:
        script_path = base_dir / script_name
        print(f"\n=== Running {script_name} ===")
        subprocess.run(
            [
                sys.executable,
                str(script_path),
                "--csv_path", args.csv_path,
                "--out_dir", str(out_dir),
            ],
            check=True
        )

    print("\nAll audits completed.")
    print(f"Artifacts saved to: {out_dir}")


if __name__ == "__main__":
    main()

Writing run_all_audits.py


In [20]:
%%writefile preprocessing.py
import re
from typing import List, Tuple

import numpy as np
import pandas as pd


CANONICAL_COLUMNS = [
    "name",
    "country_origin",
    "domain",
    "price_category",
    "founded",
    "presence_world",
    "presence_russia",
    "presence_regions",
    "description",
    "plans",
    "total_rented_area",
]

REQ_FEATURES = [
    "name",
    "description",
    "price_category",
    "country_origin",
    "domain",
    "presence_world",
    "presence_russia",
    "presence_regions",
    "plans",
    "founded",
]

BASE_FEATURE_COLUMNS = [
    "name",
    "name_clean",
    "description",
    "price_category",
    "country_origin",
    "domain",
    "presence_world",
    "presence_russia",
    "presence_own",
    "presence_franchise",
    "presence_regions",
    "n_regions",
    "plans",
    "founded",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


COUNTRY_MAPPING = {
    "россия": "Россия",
    "росия": "Россия",
    "россии": "Россия",
    "республика беларусь": "Республика Беларусь",
    "беларусь": "Республика Беларусь",
    "usa": "США",
    "америка": "США",
    "сша": "США",
}

LEGAL_FORMS = ["ооо", "зао", "ип", "оао", "ooo", "zao"]
LEGAL_FORMS_PATTERN = r"\b(?:%s)\b" % "|".join(LEGAL_FORMS)


def harmonize_schema(df: pd.DataFrame) -> pd.DataFrame:
    """
    Приводит схему к каноническому виду.
    Исправляет contry_origin -> country_origin.
    """
    df = df.copy()

    if "country_origin" not in df.columns and "contry_origin" in df.columns:
        df = df.rename(columns={"contry_origin": "country_origin"})

    if "country_origin" in df.columns and "contry_origin" in df.columns:
        df = df.drop(columns=["contry_origin"])

    return df


def _normalize_text(s: object) -> str:
    """
    Базовая мягкая нормализация текста.
    """
    if pd.isna(s):
        return ""

    s = str(s).strip().lower()
    s = s.replace("\u200b", "")
    s = s.replace("\xa0", " ")
    s = re.sub(r"\s+", " ", s)
    return s


def _deduplicate_tokens_preserve_order(tokens: List[str]) -> List[str]:
    seen = set()
    out = []

    for token in tokens:
        if token not in seen:
            out.append(token)
            seen.add(token)

    return out


def normalize_name(s: object) -> str:
    """
    Нормализация названия бренда:
    - lower
    - удаление спецсимволов
    - удаление ОПФ
    - удаление повторяющихся токенов
    """
    s = _normalize_text(s)
    s = re.sub(r"[^a-zа-я0-9 ]", " ", s)
    s = re.sub(LEGAL_FORMS_PATTERN, " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    tokens = [t for t in s.split() if t]
    tokens = _deduplicate_tokens_preserve_order(tokens)

    return " ".join(tokens)


def normalize_country(s: object) -> str:
    """
    Нормализация страны происхождения.
    """
    raw = _normalize_text(s)
    raw = re.sub(r"[^\w\s,-]", "", raw)
    raw = re.sub(r"\s+", " ", raw).strip()

    if not raw:
        return "Неизвестно"

    return COUNTRY_MAPPING.get(raw, str(s).strip())


def normalize_domain(s: object) -> str:
    """
    Мягкая нормализация domain.
    Не делаем агрессивного переписывания классов.
    """
    if pd.isna(s):
        return "Неизвестно"

    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)

    return s if s else "Неизвестно"


def normalize_price_category(s: object) -> str:
    """
    Нормализация price_category:
    - lower
    - удаление мусорных пробелов
    - замена , и : на ;
    - удаление дублей лейблов с сохранением порядка
    """
    if pd.isna(s):
        return "неизвестно"

    s = str(s).strip().lower()
    s = s.replace("\u200b", "")
    s = s.replace("\xa0", " ")
    s = s.replace(",", ";")
    s = s.replace(":", ";")

    parts = [p.strip() for p in s.split(";") if p.strip()]
    parts = _deduplicate_tokens_preserve_order(parts)

    return ";".join(parts) if parts else "неизвестно"


def extract_presence_values(text: object) -> Tuple[float, float]:
    """
    Извлекает из presence_russia:
    - presence_own
    - presence_franchise

    Примеры:
    '82 и 20 франчайзинговых' -> (82, 20)
    '21' -> (21, 0)
    """
    if pd.isna(text):
        return np.nan, np.nan

    text = str(text)
    numbers = list(map(int, re.findall(r"\d+", text)))
    low = text.lower()

    if "франчайз" in low:
        if len(numbers) >= 2:
            return float(numbers[0]), float(numbers[1])
        if len(numbers) == 1:
            return 0.0, float(numbers[0])

    if len(numbers) >= 1:
        return float(numbers[0]), 0.0

    return np.nan, np.nan


def region_count(text: object) -> int:
    """
    Считает количество регионов в presence_regions.
    """
    if pd.isna(text):
        return 0

    return len([p.strip() for p in str(text).split(";") if p.strip()])


def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Главная функция общей подготовки:
    - harmonize schema
    - cleaning / normalization
    - feature engineering
    """
    df = harmonize_schema(df).copy()

    # -------- text normalization --------
    if "name" in df.columns:
        df["name_clean"] = df["name"].apply(normalize_name)

    if "country_origin" in df.columns:
        df["country_origin"] = df["country_origin"].apply(normalize_country)

    if "domain" in df.columns:
        df["domain"] = df["domain"].apply(normalize_domain)

    if "price_category" in df.columns:
        df["price_category"] = df["price_category"].apply(normalize_price_category)

    # -------- numeric parsing --------
    for col in ["presence_world", "plans", "founded", "total_rented_area"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # -------- engineered features --------
    if "presence_russia" in df.columns:
        extracted = df["presence_russia"].apply(extract_presence_values)
        df["presence_own"] = extracted.apply(lambda x: x[0])
        df["presence_franchise"] = extracted.apply(lambda x: x[1])

    if "presence_regions" in df.columns:
        df["n_regions"] = df["presence_regions"].apply(region_count)

    if "description" in df.columns:
        desc = df["description"].fillna("").astype(str)
        df["desc_len"] = desc.str.len()
        df["desc_has_digits"] = desc.str.contains(r"\d", regex=True).astype(int)
        df["desc_year_mentions"] = desc.str.count(r"\b(?:18|19|20)\d{2}\b")

    if "total_rented_area" in df.columns:
        df["has_total_rented_area"] = df["total_rented_area"].notna().astype(int)

    return df


def features_for_model(df: pd.DataFrame, target: str = "") -> pd.DataFrame:
    """
    Готовит датасет для модели и при необходимости убирает target.
    """
    df = prepare_features(df)

    if target and target in df.columns:
        df = df.drop(columns=[target], errors="ignore")

    return df


def ensure_required_columns(df: pd.DataFrame, required: List[str]) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def get_base_feature_columns() -> List[str]:
    return BASE_FEATURE_COLUMNS.copy()

Overwriting preprocessing.py


In [24]:
%%writefile train_domain.py
import argparse
import json
from pathlib import Path

import joblib
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

from preprocessing import prepare_features, ensure_required_columns

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False


RANDOM_STATE = 42


REQUIRED_COLUMNS = [
    "name",
    "name_clean",
    "description",
    "price_category",
    "country_origin",
    "domain",
    "presence_world",
    "presence_russia",
    "presence_own",
    "presence_franchise",
    "presence_regions",
    "n_regions",
    "plans",
    "founded",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


FEATURE_COLUMNS = [
    "description",
    "name_clean",
    "price_category",
    "country_origin",
    "founded",
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


NUMERIC_FEATURES = [
    "founded",
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


CATEGORICAL_FEATURES = [
    "country_origin",
    "price_category",
]


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def build_preprocessor() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                    ("scaler", StandardScaler()),
                ]),
                NUMERIC_FEATURES,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="constant", fill_value="неизвестно")),
                    ("ohe", make_ohe()),
                ]),
                CATEGORICAL_FEATURES,
            ),
            (
                "desc_tfidf",
                TfidfVectorizer(
                    max_features=4000,
                    ngram_range=(1, 2),
                    min_df=2,
                ),
                "description",
            ),
            (
                "name_tfidf",
                TfidfVectorizer(
                    max_features=1500,
                    ngram_range=(1, 2),
                    min_df=2,
                ),
                "name_clean",
            ),
        ],
        remainder="drop",
    )


def load_and_prepare_data(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = prepare_features(df)

    ensure_required_columns(df, REQUIRED_COLUMNS)

    df = df[df["domain"].notna()].copy()
    df["domain"] = df["domain"].astype(str).str.strip()
    df = df[df["domain"] != ""].copy()
    df = df[df["domain"] != "Неизвестно"].copy()

    class_counts = df["domain"].value_counts()
    keep_classes = class_counts[class_counts >= 5].index
    df = df[df["domain"].isin(keep_classes)].copy()

    df["description"] = df["description"].fillna("").astype(str)
    df["name_clean"] = df["name_clean"].fillna("").astype(str)
    df["country_origin"] = df["country_origin"].fillna("Неизвестно").astype(str)
    df["price_category"] = df["price_category"].fillna("неизвестно").astype(str)

    return df


def build_candidates(n_classes: int):
    preprocessor = build_preprocessor()

    candidates = [
        {
            "name": "DummyMostFrequent",
            "family": "baseline",
            "pipeline": Pipeline([
                ("preprocessor", preprocessor),
                ("model", DummyClassifier(strategy="most_frequent")),
            ]),
            "param_grid": None,
            "needs_label_encoding": False,
        },
        {
            "name": "LogisticRegression",
            "family": "linear",
            "pipeline": Pipeline([
                ("preprocessor", preprocessor),
                ("model", LogisticRegression(
                    max_iter=3000,
                    class_weight="balanced",
                    solver="saga",
                    n_jobs=-1,
                    random_state=RANDOM_STATE,
                )),
            ]),
            "param_grid": {
                "model__C": [0.3, 1.0, 3.0],
            },
            "needs_label_encoding": False,
        },
        {
            "name": "RandomForest",
            "family": "bagging",
            "pipeline": Pipeline([
                ("preprocessor", preprocessor),
                ("model", RandomForestClassifier(
                    n_estimators=400,
                    class_weight="balanced_subsample",
                    n_jobs=-1,
                    random_state=RANDOM_STATE,
                )),
            ]),
            "param_grid": {
                "model__max_depth": [None, 20],
                "model__min_samples_leaf": [1, 2],
            },
            "needs_label_encoding": False,
        },
    ]

    if HAS_XGBOOST:
        candidates.append(
            {
                "name": "XGBoost",
                "family": "boosting",
                "pipeline": Pipeline([
                    ("preprocessor", preprocessor),
                    ("model", XGBClassifier(
                        objective="multi:softprob",
                        num_class=n_classes,
                        n_estimators=300,
                        learning_rate=0.1,
                        max_depth=4,
                        subsample=0.9,
                        colsample_bytree=0.9,
                        reg_lambda=1.0,
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                        eval_metric="mlogloss",
                    )),
                ]),
                "param_grid": {
                    "model__learning_rate": [0.05, 0.1],
                    "model__max_depth": [4, 6],
                },
                "needs_label_encoding": True,
            }
        )

    return candidates


def evaluate_candidate(candidate, X_train, y_train, X_test, y_test, cv):
    pipe = clone(candidate["pipeline"])

    if candidate["needs_label_encoding"]:
        le = LabelEncoder()
        y_train_fit = le.fit_transform(y_train)
        y_test_true = y_test.astype(str).values
    else:
        le = None
        y_train_fit = y_train
        y_test_true = y_test.astype(str).values

    if candidate["param_grid"]:
        search = GridSearchCV(
            estimator=pipe,
            param_grid=candidate["param_grid"],
            scoring="f1_macro",
            cv=cv,
            n_jobs=-1,
            verbose=1,
            refit=True,
        )
        search.fit(X_train, y_train_fit)
        fitted_model = search.best_estimator_
        best_params = search.best_params_
        cv_best_score = float(search.best_score_)
    else:
        fitted_model = pipe.fit(X_train, y_train_fit)
        best_params = {}
        cv_best_score = None

    y_pred_raw = fitted_model.predict(X_test)

    if le is not None:
        y_pred = le.inverse_transform(y_pred_raw)
    else:
        y_pred = y_pred_raw

    acc = float(accuracy_score(y_test_true, y_pred))
    macro_f1 = float(f1_score(y_test_true, y_pred, average="macro"))
    weighted_f1 = float(f1_score(y_test_true, y_pred, average="weighted"))
    report_text = classification_report(y_test_true, y_pred, zero_division=0)

    return {
        "name": candidate["name"],
        "family": candidate["family"],
        "model": fitted_model,
        "best_params": best_params,
        "cv_best_macro_f1": cv_best_score,
        "holdout_accuracy": acc,
        "holdout_macro_f1": macro_f1,
        "holdout_weighted_f1": weighted_f1,
        "classification_report": report_text,
        "label_encoder": le,
    }


def save_artifacts(
    out_dir: Path,
    winner: dict,
    leaderboard: pd.DataFrame,
    classes: list,
    rows_used: int,
    feature_columns: list,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    leaderboard.to_csv(out_dir / "leaderboard.csv", index=False, encoding="utf-8-sig")
    joblib.dump(winner["model"], out_dir / "domain_model.joblib")

    if winner["label_encoder"] is not None:
        joblib.dump(winner["label_encoder"], out_dir / "domain_label_encoder.joblib")

    report = {
        "target": "domain",
        "winner_name": winner["name"],
        "winner_family": winner["family"],
        "rows_used": int(rows_used),
        "n_classes": int(len(classes)),
        "feature_columns": feature_columns,
        "best_params": winner["best_params"],
        "cv_best_macro_f1": winner["cv_best_macro_f1"],
        "holdout_accuracy": winner["holdout_accuracy"],
        "holdout_macro_f1": winner["holdout_macro_f1"],
        "holdout_weighted_f1": winner["holdout_weighted_f1"],
    }

    with open(out_dir / "domain_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    with open(out_dir / "domain_classes.json", "w", encoding="utf-8") as f:
        json.dump(sorted(classes), f, ensure_ascii=False, indent=2)

    with open(out_dir / "domain_classification_report.txt", "w", encoding="utf-8") as f:
        f.write(winner["classification_report"])


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--out_dir",
        type=str,
        default="/kaggle/working/artifacts/domain",
    )
    parser.add_argument(
        "--test_size",
        type=float,
        default=0.2,
    )
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    df = load_and_prepare_data(args.csv_path)

    X = df[FEATURE_COLUMNS].copy()
    y = df["domain"].astype(str)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=args.test_size,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    candidates = build_candidates(n_classes=y.nunique())

    all_results = []

    for candidate in candidates:
        print("\n==============================")
        print(f"Training: {candidate['name']} [{candidate['family']}]")
        print("==============================")

        result = evaluate_candidate(candidate, X_train, y_train, X_test, y_test, cv)
        all_results.append(result)

        print(f"Accuracy      : {result['holdout_accuracy']:.4f}")
        print(f"Macro F1      : {result['holdout_macro_f1']:.4f}")
        print(f"Weighted F1   : {result['holdout_weighted_f1']:.4f}")
        print(result["classification_report"])

    leaderboard = pd.DataFrame([
        {
            "name": r["name"],
            "family": r["family"],
            "cv_best_macro_f1": r["cv_best_macro_f1"],
            "holdout_macro_f1": r["holdout_macro_f1"],
            "holdout_accuracy": r["holdout_accuracy"],
            "holdout_weighted_f1": r["holdout_weighted_f1"],
        }
        for r in all_results
    ]).sort_values(
        by=["holdout_macro_f1", "holdout_accuracy"],
        ascending=[False, False],
    ).reset_index(drop=True)

    winner_name = leaderboard.iloc[0]["name"]
    winner = next(r for r in all_results if r["name"] == winner_name)

    print("\n==============================")
    print("FINAL LEADERBOARD")
    print("==============================")
    print(leaderboard[["name", "family", "holdout_macro_f1", "holdout_accuracy"]])

    winner_print = {
        "name": winner["name"],
        "family": winner["family"],
        "best_params": winner["best_params"],
        "cv_best_macro_f1": winner["cv_best_macro_f1"],
        "holdout_accuracy": winner["holdout_accuracy"],
        "holdout_macro_f1": winner["holdout_macro_f1"],
        "holdout_weighted_f1": winner["holdout_weighted_f1"],
    }

    print("\nWinner:")
    print(json.dumps(winner_print, ensure_ascii=False, indent=2))

    save_artifacts(
        out_dir=out_dir,
        winner=winner,
        leaderboard=leaderboard,
        classes=sorted(y.unique().tolist()),
        rows_used=len(df),
        feature_columns=FEATURE_COLUMNS,
    )

    print(f"\nArtifacts saved to: {out_dir}")


if __name__ == "__main__":
    main()

Overwriting train_domain.py


In [25]:
import argparse
import json
from pathlib import Path

import joblib
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

from preprocessing import prepare_features, ensure_required_columns

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False


RANDOM_STATE = 42


REQUIRED_COLUMNS = [
    "name",
    "name_clean",
    "description",
    "price_category",
    "country_origin",
    "domain",
    "presence_world",
    "presence_russia",
    "presence_own",
    "presence_franchise",
    "presence_regions",
    "n_regions",
    "plans",
    "founded",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


FEATURE_COLUMNS = [
    "description",
    "name_clean",
    "price_category",
    "country_origin",
    "founded",
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


NUMERIC_FEATURES = [
    "founded",
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


CATEGORICAL_FEATURES = [
    "country_origin",
    "price_category",
]


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def build_preprocessor() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                    ("scaler", StandardScaler()),
                ]),
                NUMERIC_FEATURES,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="constant", fill_value="неизвестно")),
                    ("ohe", make_ohe()),
                ]),
                CATEGORICAL_FEATURES,
            ),
            (
                "desc_tfidf",
                TfidfVectorizer(
                    max_features=4000,
                    ngram_range=(1, 2),
                    min_df=2,
                ),
                "description",
            ),
            (
                "name_tfidf",
                TfidfVectorizer(
                    max_features=1500,
                    ngram_range=(1, 2),
                    min_df=2,
                ),
                "name_clean",
            ),
        ],
        remainder="drop",
    )


def load_and_prepare_data(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = prepare_features(df)

    ensure_required_columns(df, REQUIRED_COLUMNS)

    df = df[df["domain"].notna()].copy()
    df["domain"] = df["domain"].astype(str).str.strip()
    df = df[df["domain"] != ""].copy()
    df = df[df["domain"] != "Неизвестно"].copy()

    class_counts = df["domain"].value_counts()
    keep_classes = class_counts[class_counts >= 5].index
    df = df[df["domain"].isin(keep_classes)].copy()

    df["description"] = df["description"].fillna("").astype(str)
    df["name_clean"] = df["name_clean"].fillna("").astype(str)
    df["country_origin"] = df["country_origin"].fillna("Неизвестно").astype(str)
    df["price_category"] = df["price_category"].fillna("неизвестно").astype(str)

    return df


def build_candidates(n_classes: int):
    preprocessor = build_preprocessor()

    candidates = [
        {
            "name": "DummyMostFrequent",
            "family": "baseline",
            "pipeline": Pipeline([
                ("preprocessor", preprocessor),
                ("model", DummyClassifier(strategy="most_frequent")),
            ]),
            "param_grid": None,
            "needs_label_encoding": False,
        },
        {
            "name": "LogisticRegression",
            "family": "linear",
            "pipeline": Pipeline([
                ("preprocessor", preprocessor),
                ("model", LogisticRegression(
                    max_iter=3000,
                    class_weight="balanced",
                    solver="saga",
                    n_jobs=-1,
                    random_state=RANDOM_STATE,
                )),
            ]),
            "param_grid": {
                "model__C": [0.3, 1.0, 3.0],
            },
            "needs_label_encoding": False,
        },
        {
            "name": "RandomForest",
            "family": "bagging",
            "pipeline": Pipeline([
                ("preprocessor", preprocessor),
                ("model", RandomForestClassifier(
                    n_estimators=400,
                    class_weight="balanced_subsample",
                    n_jobs=-1,
                    random_state=RANDOM_STATE,
                )),
            ]),
            "param_grid": {
                "model__max_depth": [None, 20],
                "model__min_samples_leaf": [1, 2],
            },
            "needs_label_encoding": False,
        },
    ]

    if HAS_XGBOOST:
        candidates.append(
            {
                "name": "XGBoost",
                "family": "boosting",
                "pipeline": Pipeline([
                    ("preprocessor", preprocessor),
                    ("model", XGBClassifier(
                        objective="multi:softprob",
                        num_class=n_classes,
                        n_estimators=300,
                        learning_rate=0.1,
                        max_depth=4,
                        subsample=0.9,
                        colsample_bytree=0.9,
                        reg_lambda=1.0,
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                        eval_metric="mlogloss",
                    )),
                ]),
                "param_grid": {
                    "model__learning_rate": [0.05, 0.1],
                    "model__max_depth": [4, 6],
                },
                "needs_label_encoding": True,
            }
        )

    return candidates


def evaluate_candidate(candidate, X_train, y_train, X_test, y_test, cv):
    pipe = clone(candidate["pipeline"])

    if candidate["needs_label_encoding"]:
        le = LabelEncoder()
        y_train_fit = le.fit_transform(y_train)
        y_test_true = y_test.astype(str).values
    else:
        le = None
        y_train_fit = y_train
        y_test_true = y_test.astype(str).values

    if candidate["param_grid"]:
        search = GridSearchCV(
            estimator=pipe,
            param_grid=candidate["param_grid"],
            scoring="f1_macro",
            cv=cv,
            n_jobs=-1,
            verbose=1,
            refit=True,
        )
        search.fit(X_train, y_train_fit)
        fitted_model = search.best_estimator_
        best_params = search.best_params_
        cv_best_score = float(search.best_score_)
    else:
        fitted_model = pipe.fit(X_train, y_train_fit)
        best_params = {}
        cv_best_score = None

    y_pred_raw = fitted_model.predict(X_test)

    if le is not None:
        y_pred = le.inverse_transform(y_pred_raw)
    else:
        y_pred = y_pred_raw

    acc = float(accuracy_score(y_test_true, y_pred))
    macro_f1 = float(f1_score(y_test_true, y_pred, average="macro"))
    weighted_f1 = float(f1_score(y_test_true, y_pred, average="weighted"))
    report_text = classification_report(y_test_true, y_pred, zero_division=0)

    return {
        "name": candidate["name"],
        "family": candidate["family"],
        "model": fitted_model,
        "best_params": best_params,
        "cv_best_macro_f1": cv_best_score,
        "holdout_accuracy": acc,
        "holdout_macro_f1": macro_f1,
        "holdout_weighted_f1": weighted_f1,
        "classification_report": report_text,
        "label_encoder": le,
    }


def save_artifacts(
    out_dir: Path,
    winner: dict,
    leaderboard: pd.DataFrame,
    classes: list,
    rows_used: int,
    feature_columns: list,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    leaderboard.to_csv(out_dir / "leaderboard.csv", index=False, encoding="utf-8-sig")
    joblib.dump(winner["model"], out_dir / "domain_model.joblib")

    if winner["label_encoder"] is not None:
        joblib.dump(winner["label_encoder"], out_dir / "domain_label_encoder.joblib")

    report = {
        "target": "domain",
        "winner_name": winner["name"],
        "winner_family": winner["family"],
        "rows_used": int(rows_used),
        "n_classes": int(len(classes)),
        "feature_columns": feature_columns,
        "best_params": winner["best_params"],
        "cv_best_macro_f1": winner["cv_best_macro_f1"],
        "holdout_accuracy": winner["holdout_accuracy"],
        "holdout_macro_f1": winner["holdout_macro_f1"],
        "holdout_weighted_f1": winner["holdout_weighted_f1"],
    }

    with open(out_dir / "domain_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    with open(out_dir / "domain_classes.json", "w", encoding="utf-8") as f:
        json.dump(sorted(classes), f, ensure_ascii=False, indent=2)

    with open(out_dir / "domain_classification_report.txt", "w", encoding="utf-8") as f:
        f.write(winner["classification_report"])


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--out_dir",
        type=str,
        default="/kaggle/working/artifacts/domain",
    )
    parser.add_argument(
        "--test_size",
        type=float,
        default=0.2,
    )
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    df = load_and_prepare_data(args.csv_path)

    X = df[FEATURE_COLUMNS].copy()
    y = df["domain"].astype(str)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=args.test_size,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    candidates = build_candidates(n_classes=y.nunique())

    all_results = []

    for candidate in candidates:
        print("\n==============================")
        print(f"Training: {candidate['name']} [{candidate['family']}]")
        print("==============================")

        result = evaluate_candidate(candidate, X_train, y_train, X_test, y_test, cv)
        all_results.append(result)

        print(f"Accuracy      : {result['holdout_accuracy']:.4f}")
        print(f"Macro F1      : {result['holdout_macro_f1']:.4f}")
        print(f"Weighted F1   : {result['holdout_weighted_f1']:.4f}")
        print(result["classification_report"])

    leaderboard = pd.DataFrame([
        {
            "name": r["name"],
            "family": r["family"],
            "cv_best_macro_f1": r["cv_best_macro_f1"],
            "holdout_macro_f1": r["holdout_macro_f1"],
            "holdout_accuracy": r["holdout_accuracy"],
            "holdout_weighted_f1": r["holdout_weighted_f1"],
        }
        for r in all_results
    ]).sort_values(
        by=["holdout_macro_f1", "holdout_accuracy"],
        ascending=[False, False],
    ).reset_index(drop=True)

    winner_name = leaderboard.iloc[0]["name"]
    winner = next(r for r in all_results if r["name"] == winner_name)

    print("\n==============================")
    print("FINAL LEADERBOARD")
    print("==============================")
    print(leaderboard[["name", "family", "holdout_macro_f1", "holdout_accuracy"]])

    winner_print = {
        "name": winner["name"],
        "family": winner["family"],
        "best_params": winner["best_params"],
        "cv_best_macro_f1": winner["cv_best_macro_f1"],
        "holdout_accuracy": winner["holdout_accuracy"],
        "holdout_macro_f1": winner["holdout_macro_f1"],
        "holdout_weighted_f1": winner["holdout_weighted_f1"],
    }

    print("\nWinner:")
    print(json.dumps(winner_print, ensure_ascii=False, indent=2))

    save_artifacts(
        out_dir=out_dir,
        winner=winner,
        leaderboard=leaderboard,
        classes=sorted(y.unique().tolist()),
        rows_used=len(df),
        feature_columns=FEATURE_COLUMNS,
    )

    print(f"\nArtifacts saved to: {out_dir}")


if __name__ == "__main__":
    main()


Training: DummyMostFrequent [baseline]
Accuracy      : 0.1413
Macro F1      : 0.0067
Weighted F1   : 0.0350
                                       precision    recall  f1-score   support

               Авто и товары для авто       0.00      0.00      0.00         8
                           Аксессуары       0.00      0.00      0.00        15
                               Аптека       0.00      0.00      0.00        11
                   Банк, кредит, заем       0.00      0.00      0.00        15
                      Бытовая техника       0.00      0.00      0.00         6
                              Вендинг       0.00      0.00      0.00         3
           Вино и алкогольные напитки       0.00      0.00      0.00        12
                         Все для дома       0.00      0.00      0.00        21
      Здоровье, лечение, профилактика       0.00      0.00      0.00        10
                   Зоотовары и услуги       0.00      0.00      0.00         6
                     

In [26]:
%%writefile train_founded.py
import argparse
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from preprocessing import prepare_features, ensure_required_columns

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False


RANDOM_STATE = 42


REQUIRED_COLUMNS = [
    "name",
    "name_clean",
    "description",
    "price_category",
    "country_origin",
    "domain",
    "presence_world",
    "presence_russia",
    "presence_own",
    "presence_franchise",
    "presence_regions",
    "n_regions",
    "plans",
    "founded",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


FEATURE_COLUMNS = [
    "description",
    "name_clean",
    "price_category",
    "country_origin",
    "domain",
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


NUMERIC_FEATURES = [
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


CATEGORICAL_FEATURES = [
    "country_origin",
    "price_category",
    "domain",
]


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def build_preprocessor() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                    ("scaler", StandardScaler()),
                ]),
                NUMERIC_FEATURES,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="constant", fill_value="неизвестно")),
                    ("ohe", make_ohe()),
                ]),
                CATEGORICAL_FEATURES,
            ),
            (
                "desc_tfidf",
                TfidfVectorizer(
                    max_features=4000,
                    ngram_range=(1, 2),
                    min_df=2,
                ),
                "description",
            ),
            (
                "name_tfidf",
                TfidfVectorizer(
                    max_features=1500,
                    ngram_range=(1, 2),
                    min_df=2,
                ),
                "name_clean",
            ),
        ],
        remainder="drop",
    )


def load_and_prepare_data(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = prepare_features(df)

    ensure_required_columns(df, REQUIRED_COLUMNS)

    df["founded"] = pd.to_numeric(df["founded"], errors="coerce")
    df = df[df["founded"].notna()].copy()
    df = df[df["founded"].between(1850, 2025)].copy()

    df["description"] = df["description"].fillna("").astype(str)
    df["name_clean"] = df["name_clean"].fillna("").astype(str)
    df["country_origin"] = df["country_origin"].fillna("Неизвестно").astype(str)
    df["price_category"] = df["price_category"].fillna("неизвестно").astype(str)
    df["domain"] = df["domain"].fillna("Неизвестно").astype(str)

    return df


def make_ttr(regressor):
    return TransformedTargetRegressor(
        regressor=regressor,
        func=np.log1p,
        inverse_func=np.expm1,
    )


def build_candidates():
    preprocessor = build_preprocessor()

    candidates = [
        {
            "name": "DummyMedian",
            "family": "baseline",
            "pipeline": Pipeline([
                ("preprocessor", preprocessor),
                ("model", DummyRegressor(strategy="median")),
            ]),
            "param_grid": None,
        },
        {
            "name": "Ridge",
            "family": "linear",
            "pipeline": Pipeline([
                ("preprocessor", preprocessor),
                ("model", make_ttr(Ridge(random_state=RANDOM_STATE))),
            ]),
            "param_grid": {
                "model__regressor__alpha": [0.3, 1.0, 3.0],
            },
        },
        {
            "name": "RandomForest",
            "family": "bagging",
            "pipeline": Pipeline([
                ("preprocessor", preprocessor),
                ("model", make_ttr(RandomForestRegressor(
                    n_estimators=400,
                    n_jobs=-1,
                    random_state=RANDOM_STATE,
                ))),
            ]),
            "param_grid": {
                "model__regressor__max_depth": [None, 12],
                "model__regressor__min_samples_leaf": [1, 2],
            },
        },
    ]

    if HAS_XGBOOST:
        candidates.append(
            {
                "name": "XGBoost",
                "family": "boosting",
                "pipeline": Pipeline([
                    ("preprocessor", preprocessor),
                    ("model", make_ttr(XGBRegressor(
                        n_estimators=300,
                        learning_rate=0.1,
                        max_depth=4,
                        subsample=0.9,
                        colsample_bytree=0.9,
                        reg_lambda=1.0,
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                        objective="reg:squarederror",
                        eval_metric="rmse",
                    ))),
                ]),
                "param_grid": {
                    "model__regressor__learning_rate": [0.05, 0.1],
                    "model__regressor__max_depth": [4, 6],
                },
            }
        )

    return candidates


def evaluate_candidate(candidate, X_train, y_train, X_test, y_test, cv):
    pipe = clone(candidate["pipeline"])

    if candidate["param_grid"]:
        search = GridSearchCV(
            estimator=pipe,
            param_grid=candidate["param_grid"],
            scoring="neg_mean_absolute_error",
            cv=cv,
            n_jobs=-1,
            verbose=1,
            refit=True,
        )
        search.fit(X_train, y_train)
        fitted_model = search.best_estimator_
        best_params = search.best_params_
        cv_best_mae = float(-search.best_score_)
    else:
        fitted_model = pipe.fit(X_train, y_train)
        best_params = {}
        cv_best_mae = None

    y_pred = fitted_model.predict(X_test)
    y_pred = np.clip(y_pred, 0, None)

    mae = float(mean_absolute_error(y_test, y_pred))
    rmse_value = rmse(y_test, y_pred)
    r2 = float(r2_score(y_test, y_pred))

    prediction_preview = pd.DataFrame({
        "y_true": y_test.values,
        "y_pred": y_pred,
        "abs_error": np.abs(y_test.values - y_pred),
    }).sort_values(by="abs_error", ascending=False)

    return {
        "name": candidate["name"],
        "family": candidate["family"],
        "model": fitted_model,
        "best_params": best_params,
        "cv_best_mae": cv_best_mae,
        "holdout_mae": mae,
        "holdout_rmse": rmse_value,
        "holdout_r2": r2,
        "prediction_preview": prediction_preview,
    }


def save_artifacts(
    out_dir: Path,
    winner: dict,
    leaderboard: pd.DataFrame,
    rows_used: int,
    feature_columns: list,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    leaderboard.to_csv(out_dir / "leaderboard.csv", index=False, encoding="utf-8-sig")
    joblib.dump(winner["model"], out_dir / "founded_model.joblib")

    report = {
        "target": "founded",
        "winner_name": winner["name"],
        "winner_family": winner["family"],
        "rows_used": int(rows_used),
        "feature_columns": feature_columns,
        "best_params": winner["best_params"],
        "cv_best_mae": winner["cv_best_mae"],
        "holdout_mae": winner["holdout_mae"],
        "holdout_rmse": winner["holdout_rmse"],
        "holdout_r2": winner["holdout_r2"],
    }

    with open(out_dir / "founded_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    winner["prediction_preview"].head(200).to_csv(
        out_dir / "founded_predictions_preview.csv",
        index=False,
        encoding="utf-8-sig",
    )


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--out_dir",
        type=str,
        default="/kaggle/working/artifacts/founded",
    )
    parser.add_argument(
        "--test_size",
        type=float,
        default=0.2,
    )
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    df = load_and_prepare_data(args.csv_path)

    X = df[FEATURE_COLUMNS].copy()
    y = df["founded"].astype(float)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=args.test_size,
        random_state=RANDOM_STATE,
    )

    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    candidates = build_candidates()

    all_results = []

    for candidate in candidates:
        print("\n==============================")
        print(f"Training: {candidate['name']} [{candidate['family']}]")
        print("==============================")

        result = evaluate_candidate(candidate, X_train, y_train, X_test, y_test, cv)
        all_results.append(result)

        print(f"MAE           : {result['holdout_mae']:.4f}")
        print(f"RMSE          : {result['holdout_rmse']:.4f}")
        print(f"R2            : {result['holdout_r2']:.4f}")

    leaderboard = pd.DataFrame([
        {
            "name": r["name"],
            "family": r["family"],
            "cv_best_mae": r["cv_best_mae"],
            "holdout_mae": r["holdout_mae"],
            "holdout_rmse": r["holdout_rmse"],
            "holdout_r2": r["holdout_r2"],
        }
        for r in all_results
    ]).sort_values(
        by=["holdout_mae", "holdout_rmse"],
        ascending=[True, True],
    ).reset_index(drop=True)

    winner_name = leaderboard.iloc[0]["name"]
    winner = next(r for r in all_results if r["name"] == winner_name)

    print("\n==============================")
    print("FINAL LEADERBOARD")
    print("==============================")
    print(leaderboard[["name", "family", "holdout_mae", "holdout_rmse", "holdout_r2"]])

    winner_print = {
        "name": winner["name"],
        "family": winner["family"],
        "best_params": winner["best_params"],
        "cv_best_mae": winner["cv_best_mae"],
        "holdout_mae": winner["holdout_mae"],
        "holdout_rmse": winner["holdout_rmse"],
        "holdout_r2": winner["holdout_r2"],
    }

    print("\nWinner:")
    print(json.dumps(winner_print, ensure_ascii=False, indent=2))

    save_artifacts(
        out_dir=out_dir,
        winner=winner,
        leaderboard=leaderboard,
        rows_used=len(df),
        feature_columns=FEATURE_COLUMNS,
    )

    print(f"\nArtifacts saved to: {out_dir}")


if __name__ == "__main__":
    main()

Writing train_founded.py


In [27]:
import argparse
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from preprocessing import prepare_features, ensure_required_columns

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False


RANDOM_STATE = 42


REQUIRED_COLUMNS = [
    "name",
    "name_clean",
    "description",
    "price_category",
    "country_origin",
    "domain",
    "presence_world",
    "presence_russia",
    "presence_own",
    "presence_franchise",
    "presence_regions",
    "n_regions",
    "plans",
    "founded",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


FEATURE_COLUMNS = [
    "description",
    "name_clean",
    "price_category",
    "country_origin",
    "domain",
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


NUMERIC_FEATURES = [
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


CATEGORICAL_FEATURES = [
    "country_origin",
    "price_category",
    "domain",
]


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def build_preprocessor() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                    ("scaler", StandardScaler()),
                ]),
                NUMERIC_FEATURES,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="constant", fill_value="неизвестно")),
                    ("ohe", make_ohe()),
                ]),
                CATEGORICAL_FEATURES,
            ),
            (
                "desc_tfidf",
                TfidfVectorizer(
                    max_features=4000,
                    ngram_range=(1, 2),
                    min_df=2,
                ),
                "description",
            ),
            (
                "name_tfidf",
                TfidfVectorizer(
                    max_features=1500,
                    ngram_range=(1, 2),
                    min_df=2,
                ),
                "name_clean",
            ),
        ],
        remainder="drop",
    )


def load_and_prepare_data(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = prepare_features(df)

    ensure_required_columns(df, REQUIRED_COLUMNS)

    df["founded"] = pd.to_numeric(df["founded"], errors="coerce")
    df = df[df["founded"].notna()].copy()
    df = df[df["founded"].between(1850, 2025)].copy()

    df["description"] = df["description"].fillna("").astype(str)
    df["name_clean"] = df["name_clean"].fillna("").astype(str)
    df["country_origin"] = df["country_origin"].fillna("Неизвестно").astype(str)
    df["price_category"] = df["price_category"].fillna("неизвестно").astype(str)
    df["domain"] = df["domain"].fillna("Неизвестно").astype(str)

    return df


def make_ttr(regressor):
    return TransformedTargetRegressor(
        regressor=regressor,
        func=np.log1p,
        inverse_func=np.expm1,
    )


def build_candidates():
    preprocessor = build_preprocessor()

    candidates = [
        {
            "name": "DummyMedian",
            "family": "baseline",
            "pipeline": Pipeline([
                ("preprocessor", preprocessor),
                ("model", DummyRegressor(strategy="median")),
            ]),
            "param_grid": None,
        },
        {
            "name": "Ridge",
            "family": "linear",
            "pipeline": Pipeline([
                ("preprocessor", preprocessor),
                ("model", make_ttr(Ridge(random_state=RANDOM_STATE))),
            ]),
            "param_grid": {
                "model__regressor__alpha": [0.3, 1.0, 3.0],
            },
        },
        {
            "name": "RandomForest",
            "family": "bagging",
            "pipeline": Pipeline([
                ("preprocessor", preprocessor),
                ("model", make_ttr(RandomForestRegressor(
                    n_estimators=400,
                    n_jobs=-1,
                    random_state=RANDOM_STATE,
                ))),
            ]),
            "param_grid": {
                "model__regressor__max_depth": [None, 12],
                "model__regressor__min_samples_leaf": [1, 2],
            },
        },
    ]

    if HAS_XGBOOST:
        candidates.append(
            {
                "name": "XGBoost",
                "family": "boosting",
                "pipeline": Pipeline([
                    ("preprocessor", preprocessor),
                    ("model", make_ttr(XGBRegressor(
                        n_estimators=300,
                        learning_rate=0.1,
                        max_depth=4,
                        subsample=0.9,
                        colsample_bytree=0.9,
                        reg_lambda=1.0,
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                        objective="reg:squarederror",
                        eval_metric="rmse",
                    ))),
                ]),
                "param_grid": {
                    "model__regressor__learning_rate": [0.05, 0.1],
                    "model__regressor__max_depth": [4, 6],
                },
            }
        )

    return candidates


def evaluate_candidate(candidate, X_train, y_train, X_test, y_test, cv):
    pipe = clone(candidate["pipeline"])

    if candidate["param_grid"]:
        search = GridSearchCV(
            estimator=pipe,
            param_grid=candidate["param_grid"],
            scoring="neg_mean_absolute_error",
            cv=cv,
            n_jobs=-1,
            verbose=1,
            refit=True,
        )
        search.fit(X_train, y_train)
        fitted_model = search.best_estimator_
        best_params = search.best_params_
        cv_best_mae = float(-search.best_score_)
    else:
        fitted_model = pipe.fit(X_train, y_train)
        best_params = {}
        cv_best_mae = None

    y_pred = fitted_model.predict(X_test)
    y_pred = np.clip(y_pred, 0, None)

    mae = float(mean_absolute_error(y_test, y_pred))
    rmse_value = rmse(y_test, y_pred)
    r2 = float(r2_score(y_test, y_pred))

    prediction_preview = pd.DataFrame({
        "y_true": y_test.values,
        "y_pred": y_pred,
        "abs_error": np.abs(y_test.values - y_pred),
    }).sort_values(by="abs_error", ascending=False)

    return {
        "name": candidate["name"],
        "family": candidate["family"],
        "model": fitted_model,
        "best_params": best_params,
        "cv_best_mae": cv_best_mae,
        "holdout_mae": mae,
        "holdout_rmse": rmse_value,
        "holdout_r2": r2,
        "prediction_preview": prediction_preview,
    }


def save_artifacts(
    out_dir: Path,
    winner: dict,
    leaderboard: pd.DataFrame,
    rows_used: int,
    feature_columns: list,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    leaderboard.to_csv(out_dir / "leaderboard.csv", index=False, encoding="utf-8-sig")
    joblib.dump(winner["model"], out_dir / "founded_model.joblib")

    report = {
        "target": "founded",
        "winner_name": winner["name"],
        "winner_family": winner["family"],
        "rows_used": int(rows_used),
        "feature_columns": feature_columns,
        "best_params": winner["best_params"],
        "cv_best_mae": winner["cv_best_mae"],
        "holdout_mae": winner["holdout_mae"],
        "holdout_rmse": winner["holdout_rmse"],
        "holdout_r2": winner["holdout_r2"],
    }

    with open(out_dir / "founded_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    winner["prediction_preview"].head(200).to_csv(
        out_dir / "founded_predictions_preview.csv",
        index=False,
        encoding="utf-8-sig",
    )


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--out_dir",
        type=str,
        default="/kaggle/working/artifacts/founded",
    )
    parser.add_argument(
        "--test_size",
        type=float,
        default=0.2,
    )
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    df = load_and_prepare_data(args.csv_path)

    X = df[FEATURE_COLUMNS].copy()
    y = df["founded"].astype(float)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=args.test_size,
        random_state=RANDOM_STATE,
    )

    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    candidates = build_candidates()

    all_results = []

    for candidate in candidates:
        print("\n==============================")
        print(f"Training: {candidate['name']} [{candidate['family']}]")
        print("==============================")

        result = evaluate_candidate(candidate, X_train, y_train, X_test, y_test, cv)
        all_results.append(result)

        print(f"MAE           : {result['holdout_mae']:.4f}")
        print(f"RMSE          : {result['holdout_rmse']:.4f}")
        print(f"R2            : {result['holdout_r2']:.4f}")

    leaderboard = pd.DataFrame([
        {
            "name": r["name"],
            "family": r["family"],
            "cv_best_mae": r["cv_best_mae"],
            "holdout_mae": r["holdout_mae"],
            "holdout_rmse": r["holdout_rmse"],
            "holdout_r2": r["holdout_r2"],
        }
        for r in all_results
    ]).sort_values(
        by=["holdout_mae", "holdout_rmse"],
        ascending=[True, True],
    ).reset_index(drop=True)

    winner_name = leaderboard.iloc[0]["name"]
    winner = next(r for r in all_results if r["name"] == winner_name)

    print("\n==============================")
    print("FINAL LEADERBOARD")
    print("==============================")
    print(leaderboard[["name", "family", "holdout_mae", "holdout_rmse", "holdout_r2"]])

    winner_print = {
        "name": winner["name"],
        "family": winner["family"],
        "best_params": winner["best_params"],
        "cv_best_mae": winner["cv_best_mae"],
        "holdout_mae": winner["holdout_mae"],
        "holdout_rmse": winner["holdout_rmse"],
        "holdout_r2": winner["holdout_r2"],
    }

    print("\nWinner:")
    print(json.dumps(winner_print, ensure_ascii=False, indent=2))

    save_artifacts(
        out_dir=out_dir,
        winner=winner,
        leaderboard=leaderboard,
        rows_used=len(df),
        feature_columns=FEATURE_COLUMNS,
    )

    print(f"\nArtifacts saved to: {out_dir}")


if __name__ == "__main__":
    main()


Training: DummyMedian [baseline]
MAE           : 11.2356
RMSE          : 20.0739
R2            : -0.0531

Training: Ridge [linear]
Fitting 5 folds for each of 3 candidates, totalling 15 fits
MAE           : 9.2428
RMSE          : 16.0914
R2            : 0.3233

Training: RandomForest [bagging]
Fitting 5 folds for each of 4 candidates, totalling 20 fits
MAE           : 8.8452
RMSE          : 15.8690
R2            : 0.3419

Training: XGBoost [boosting]
Fitting 5 folds for each of 4 candidates, totalling 20 fits
MAE           : 9.2679
RMSE          : 16.5552
R2            : 0.2837

FINAL LEADERBOARD
           name    family  holdout_mae  holdout_rmse  holdout_r2
0  RandomForest   bagging     8.845231     15.868996    0.341892
1         Ridge    linear     9.242787     16.091389    0.323317
2       XGBoost  boosting     9.267889     16.555228    0.283743
3   DummyMedian  baseline    11.235644     20.073923   -0.053084

Winner:
{
  "name": "RandomForest",
  "family": "bagging",
  "best_pa

In [32]:
%%writefile train_price_category.py
import argparse
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, StandardScaler

from preprocessing import prepare_features, ensure_required_columns

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False


RANDOM_STATE = 42


REQUIRED_COLUMNS = [
    "name",
    "name_clean",
    "description",
    "price_category",
    "country_origin",
    "domain",
    "presence_world",
    "presence_russia",
    "presence_own",
    "presence_franchise",
    "presence_regions",
    "n_regions",
    "plans",
    "founded",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


FEATURE_COLUMNS = [
    "description",
    "name_clean",
    "country_origin",
    "domain",
    "founded",
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


NUMERIC_FEATURES = [
    "founded",
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


CATEGORICAL_FEATURES = [
    "country_origin",
    "domain",
]


PRICE_LABEL_ORDER = [
    "дисконт",
    "ниже среднего",
    "средний",
    "выше среднего",
    "люкс / премиум",
    "неизвестно",
]


class MultiLabelMostFrequentDummy(BaseEstimator):
    def fit(self, X, y):
        self.label_frequencies_ = np.mean(y, axis=0)
        self.most_frequent_label_idx_ = int(np.argmax(self.label_frequencies_))
        self.n_labels_ = y.shape[1]
        return self

    def predict(self, X):
        n_rows = X.shape[0]
        out = np.zeros((n_rows, self.n_labels_), dtype=int)
        out[:, self.most_frequent_label_idx_] = 1
        return out

    def predict_proba(self, X):
        n_rows = X.shape[0]
        probs = np.tile(self.label_frequencies_, (n_rows, 1))
        return probs


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def normalize_price_string(s: object) -> str:
    if pd.isna(s):
        return "неизвестно"

    s = str(s).strip().lower()
    s = s.replace("\u200b", "")
    s = s.replace("\xa0", " ")
    s = s.replace(",", ";")
    s = s.replace(":", ";")

    parts = [p.strip() for p in s.split(";") if p.strip()]
    parts = list(dict.fromkeys(parts))

    return ";".join(parts) if parts else "неизвестно"


def split_price_labels(s: str):
    parts = [p.strip() for p in str(s).split(";") if p.strip()]
    return parts if parts else ["неизвестно"]


def make_multilabel_target(series: pd.Series):
    normalized = series.apply(normalize_price_string)
    label_lists = normalized.apply(split_price_labels)

    mlb = MultiLabelBinarizer(classes=PRICE_LABEL_ORDER)
    y = mlb.fit_transform(label_lists)

    return normalized, label_lists, mlb, y


def build_preprocessor() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                    ("scaler", StandardScaler()),
                ]),
                NUMERIC_FEATURES,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="constant", fill_value="неизвестно")),
                    ("ohe", make_ohe()),
                ]),
                CATEGORICAL_FEATURES,
            ),
            (
                "desc_tfidf",
                TfidfVectorizer(
                    max_features=4000,
                    ngram_range=(1, 2),
                    min_df=2,
                ),
                "description",
            ),
            (
                "name_tfidf",
                TfidfVectorizer(
                    max_features=1500,
                    ngram_range=(1, 2),
                    min_df=2,
                ),
                "name_clean",
            ),
        ],
        remainder="drop",
    )


def load_and_prepare_data(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = prepare_features(df)

    ensure_required_columns(df, REQUIRED_COLUMNS)

    df["description"] = df["description"].fillna("").astype(str)
    df["name_clean"] = df["name_clean"].fillna("").astype(str)
    df["country_origin"] = df["country_origin"].fillna("Неизвестно").astype(str)
    df["domain"] = df["domain"].fillna("Неизвестно").astype(str)
    df["price_category"] = df["price_category"].fillna("неизвестно").astype(str)

    return df


def build_candidates():
    preprocessor = build_preprocessor()

    candidates = [
        {
            "name": "DummyMostFrequent",
            "family": "baseline",
            "pipeline": Pipeline([
                ("preprocessor", preprocessor),
                ("model", MultiLabelMostFrequentDummy()),
            ]),
            "param_grid": None,
        },
        {
            "name": "LogisticRegression",
            "family": "linear",
            "pipeline": Pipeline([
                ("preprocessor", preprocessor),
                ("model", OneVsRestClassifier(
                    LogisticRegression(
                        max_iter=3000,
                        class_weight="balanced",
                        solver="liblinear",
                        random_state=RANDOM_STATE,
                    )
                )),
            ]),
            "param_grid": {
                "model__estimator__C": [0.3, 1.0, 3.0],
            },
        },
        {
            "name": "RandomForest",
            "family": "bagging",
            "pipeline": Pipeline([
                ("preprocessor", preprocessor),
                ("model", OneVsRestClassifier(
                    RandomForestClassifier(
                        n_estimators=400,
                        class_weight="balanced_subsample",
                        n_jobs=-1,
                        random_state=RANDOM_STATE,
                    )
                )),
            ]),
            "param_grid": {
                "model__estimator__max_depth": [12, 20],
                "model__estimator__min_samples_leaf": [1, 2],
            },
        },
    ]

    if HAS_XGBOOST:
        candidates.append(
            {
                "name": "XGBoost",
                "family": "boosting",
                "pipeline": Pipeline([
                    ("preprocessor", preprocessor),
                    ("model", OneVsRestClassifier(
                        XGBClassifier(
                            n_estimators=300,
                            learning_rate=0.1,
                            max_depth=4,
                            subsample=0.9,
                            colsample_bytree=0.9,
                            reg_lambda=1.0,
                            random_state=RANDOM_STATE,
                            n_jobs=-1,
                            eval_metric="logloss",
                        )
                    )),
                ]),
                "param_grid": {
                    "model__estimator__learning_rate": [0.05, 0.1],
                    "model__estimator__max_depth": [4, 6],
                },
            }
        )

    return candidates


def optimize_thresholds(y_true: np.ndarray, y_prob: np.ndarray, labels: list) -> dict:
    thresholds = {}

    for i, label in enumerate(labels):
        best_thr = 0.5
        best_f1 = -1.0

        for thr in np.arange(0.10, 0.91, 0.05):
            pred = (y_prob[:, i] >= thr).astype(int)
            score = f1_score(y_true[:, i], pred, zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_thr = float(np.round(thr, 2))

        thresholds[label] = best_thr

    return thresholds


def apply_thresholds(y_prob: np.ndarray, thresholds: dict, labels: list) -> np.ndarray:
    out = np.zeros_like(y_prob, dtype=int)

    for i, label in enumerate(labels):
        thr = thresholds[label]
        out[:, i] = (y_prob[:, i] >= thr).astype(int)

    row_sums = out.sum(axis=1)
    empty_rows = np.where(row_sums == 0)[0]

    if len(empty_rows) > 0:
        max_idx = np.argmax(y_prob[empty_rows], axis=1)
        out[empty_rows, max_idx] = 1

    return out


def multilabel_metrics_dict(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "samples_f1": float(f1_score(y_true, y_pred, average="samples", zero_division=0)),
    }


def classification_report_text(y_true: np.ndarray, y_pred: np.ndarray, labels: list) -> str:
    return classification_report(
        y_true,
        y_pred,
        target_names=labels,
        zero_division=0,
    )


def get_probabilities(fitted_model, X) -> np.ndarray:
    """
    ВАЖНО:
    predict_proba вызываем у всего pipeline, а не у внутренней модели,
    чтобы сначала отработал preprocessor.
    """
    if hasattr(fitted_model, "predict_proba"):
        probs = fitted_model.predict_proba(X)

        if isinstance(probs, list):
            cols = []
            for p in probs:
                p = np.asarray(p)
                if p.ndim == 2 and p.shape[1] == 2:
                    cols.append(p[:, 1])
                else:
                    cols.append(p.reshape(-1))
            return np.column_stack(cols)

        if isinstance(probs, np.ndarray):
            if probs.ndim == 3 and probs.shape[2] == 2:
                return probs[:, :, 1]
            return probs

    preds = fitted_model.predict(X)
    return np.asarray(preds, dtype=float)


def cross_validated_threshold_score(candidate, X_train, y_train, labels, n_splits=5):
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    fold_scores = []

    for fold_idx, (tr_idx, va_idx) in enumerate(cv.split(X_train), start=1):
        X_tr = X_train.iloc[tr_idx]
        X_va = X_train.iloc[va_idx]
        y_tr = y_train[tr_idx]
        y_va = y_train[va_idx]

        model = clone(candidate["pipeline"])
        model.fit(X_tr, y_tr)

        y_prob = get_probabilities(model, X_va)
        thresholds = optimize_thresholds(y_va, y_prob, labels)
        y_pred_thr = apply_thresholds(y_prob, thresholds, labels)

        score = f1_score(y_va, y_pred_thr, average="macro", zero_division=0)
        fold_scores.append(float(score))

        print(f"CV fold {fold_idx}: thresholded macro F1 = {score:.4f}")

    return float(np.mean(fold_scores))


def evaluate_candidate(candidate, X_train, y_train, X_test, y_test, labels, inner_cv):
    pipe = clone(candidate["pipeline"])

    if candidate["param_grid"]:
        search = GridSearchCV(
            estimator=pipe,
            param_grid=candidate["param_grid"],
            scoring="f1_macro",
            cv=inner_cv,
            n_jobs=-1,
            verbose=1,
            refit=True,
        )
        search.fit(X_train, y_train)
        fitted_model = search.best_estimator_
        best_params = search.best_params_
        cv_best_macro_f1 = float(search.best_score_)
    else:
        fitted_model = pipe.fit(X_train, y_train)
        best_params = {}
        cv_best_macro_f1 = None

    cv_threshold_macro_f1 = cross_validated_threshold_score(
        candidate=candidate,
        X_train=X_train,
        y_train=y_train,
        labels=labels,
        n_splits=5,
    )

    y_pred_direct = fitted_model.predict(X_test)
    direct_metrics = multilabel_metrics_dict(y_test, y_pred_direct)

    y_prob_train = get_probabilities(fitted_model, X_train)
    thresholds = optimize_thresholds(y_train, y_prob_train, labels)

    y_prob_test = get_probabilities(fitted_model, X_test)
    y_pred_threshold = apply_thresholds(y_prob_test, thresholds, labels)
    threshold_metrics = multilabel_metrics_dict(y_test, y_pred_threshold)

    report_text = classification_report_text(y_test, y_pred_threshold, labels)

    print("\n--- DIRECT PREDICT ---")
    print(f"Macro F1      : {direct_metrics['macro_f1']:.4f}")
    print(f"Micro F1      : {direct_metrics['micro_f1']:.4f}")
    print(f"Weighted F1   : {direct_metrics['weighted_f1']:.4f}")
    print(f"Samples F1    : {direct_metrics['samples_f1']:.4f}")

    print("\n--- THRESHOLDED PREDICT ---")
    print(f"Macro F1      : {threshold_metrics['macro_f1']:.4f}")
    print(f"Micro F1      : {threshold_metrics['micro_f1']:.4f}")
    print(f"Weighted F1   : {threshold_metrics['weighted_f1']:.4f}")
    print(f"Samples F1    : {threshold_metrics['samples_f1']:.4f}")

    print("\n--- CV DIAGNOSTICS ---")
    print(f"GridSearch CV macro F1        : {cv_best_macro_f1}")
    print(f"Manual threshold-CV macro F1  : {cv_threshold_macro_f1:.4f}")

    print("\nUsed thresholds:")
    for label in labels:
        print(f"  {label}: {thresholds[label]:.2f}")

    print("\nClassification report (thresholded predict):")
    print(report_text)

    return {
        "name": candidate["name"],
        "family": candidate["family"],
        "model": fitted_model,
        "best_params": best_params,
        "cv_best_macro_f1": cv_best_macro_f1,
        "cv_threshold_macro_f1": cv_threshold_macro_f1,
        "direct_macro_f1": direct_metrics["macro_f1"],
        "direct_micro_f1": direct_metrics["micro_f1"],
        "threshold_macro_f1": threshold_metrics["macro_f1"],
        "threshold_micro_f1": threshold_metrics["micro_f1"],
        "threshold_weighted_f1": threshold_metrics["weighted_f1"],
        "threshold_samples_f1": threshold_metrics["samples_f1"],
        "thresholds": thresholds,
        "classification_report": report_text,
    }


def save_artifacts(
    out_dir: Path,
    winner: dict,
    leaderboard: pd.DataFrame,
    labels: list,
    rows_used: int,
    feature_columns: list,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    leaderboard.to_csv(out_dir / "leaderboard.csv", index=False, encoding="utf-8-sig")
    joblib.dump(winner["model"], out_dir / "price_category_model.joblib")

    with open(out_dir / "price_category_thresholds.json", "w", encoding="utf-8") as f:
        json.dump(winner["thresholds"], f, ensure_ascii=False, indent=2)

    with open(out_dir / "price_category_labels.json", "w", encoding="utf-8") as f:
        json.dump(labels, f, ensure_ascii=False, indent=2)

    report = {
        "target": "price_category",
        "winner_name": winner["name"],
        "winner_family": winner["family"],
        "rows_used": int(rows_used),
        "feature_columns": feature_columns,
        "labels": labels,
        "best_params": winner["best_params"],
        "cv_best_macro_f1": winner["cv_best_macro_f1"],
        "cv_threshold_macro_f1": winner["cv_threshold_macro_f1"],
        "direct_macro_f1": winner["direct_macro_f1"],
        "direct_micro_f1": winner["direct_micro_f1"],
        "threshold_macro_f1": winner["threshold_macro_f1"],
        "threshold_micro_f1": winner["threshold_micro_f1"],
        "threshold_weighted_f1": winner["threshold_weighted_f1"],
        "threshold_samples_f1": winner["threshold_samples_f1"],
    }

    with open(out_dir / "price_category_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    with open(out_dir / "price_category_classification_report.txt", "w", encoding="utf-8") as f:
        f.write(winner["classification_report"])


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--out_dir",
        type=str,
        default="/kaggle/working/artifacts/price_category",
    )
    parser.add_argument(
        "--test_size",
        type=float,
        default=0.2,
    )
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    df = load_and_prepare_data(args.csv_path)

    _, _, mlb, y = make_multilabel_target(df["price_category"])
    labels = list(mlb.classes_)

    X = df[FEATURE_COLUMNS].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=args.test_size,
        random_state=RANDOM_STATE,
    )

    inner_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    candidates = build_candidates()

    all_results = []

    for candidate in candidates:
        print("\n==============================")
        print(f"Training: {candidate['name']} [{candidate['family']}]")
        print("==============================")

        result = evaluate_candidate(
            candidate=candidate,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            labels=labels,
            inner_cv=inner_cv,
        )
        all_results.append(result)

    leaderboard = pd.DataFrame([
        {
            "name": r["name"],
            "family": r["family"],
            "cv_best_macro_f1": r["cv_best_macro_f1"],
            "cv_threshold_macro_f1": r["cv_threshold_macro_f1"],
            "direct_macro_f1": r["direct_macro_f1"],
            "threshold_macro_f1": r["threshold_macro_f1"],
            "threshold_micro_f1": r["threshold_micro_f1"],
        }
        for r in all_results
    ]).sort_values(
        by=["threshold_macro_f1", "threshold_micro_f1"],
        ascending=[False, False],
    ).reset_index(drop=True)

    winner_name = leaderboard.iloc[0]["name"]
    winner = next(r for r in all_results if r["name"] == winner_name)

    print("\n==============================")
    print("FINAL LEADERBOARD")
    print("==============================")
    print(leaderboard)

    winner_print = {
        "name": winner["name"],
        "family": winner["family"],
        "best_params": winner["best_params"],
        "cv_best_macro_f1": winner["cv_best_macro_f1"],
        "cv_threshold_macro_f1": winner["cv_threshold_macro_f1"],
        "direct_macro_f1": winner["direct_macro_f1"],
        "direct_micro_f1": winner["direct_micro_f1"],
        "threshold_macro_f1": winner["threshold_macro_f1"],
        "threshold_micro_f1": winner["threshold_micro_f1"],
        "threshold_weighted_f1": winner["threshold_weighted_f1"],
        "threshold_samples_f1": winner["threshold_samples_f1"],
    }

    print("\nWinner:")
    print(json.dumps(winner_print, ensure_ascii=False, indent=2))

    save_artifacts(
        out_dir=out_dir,
        winner=winner,
        leaderboard=leaderboard,
        labels=labels,
        rows_used=len(df),
        feature_columns=FEATURE_COLUMNS,
    )

    print(f"\nArtifacts saved to: {out_dir}")


if __name__ == "__main__":
    main()

Overwriting train_price_category.py


In [33]:
import argparse
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, StandardScaler

from preprocessing import prepare_features, ensure_required_columns

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False


RANDOM_STATE = 42


REQUIRED_COLUMNS = [
    "name",
    "name_clean",
    "description",
    "price_category",
    "country_origin",
    "domain",
    "presence_world",
    "presence_russia",
    "presence_own",
    "presence_franchise",
    "presence_regions",
    "n_regions",
    "plans",
    "founded",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


FEATURE_COLUMNS = [
    "description",
    "name_clean",
    "country_origin",
    "domain",
    "founded",
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


NUMERIC_FEATURES = [
    "founded",
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


CATEGORICAL_FEATURES = [
    "country_origin",
    "domain",
]


PRICE_LABEL_ORDER = [
    "дисконт",
    "ниже среднего",
    "средний",
    "выше среднего",
    "люкс / премиум",
    "неизвестно",
]


class MultiLabelMostFrequentDummy(BaseEstimator):
    def fit(self, X, y):
        self.label_frequencies_ = np.mean(y, axis=0)
        self.most_frequent_label_idx_ = int(np.argmax(self.label_frequencies_))
        self.n_labels_ = y.shape[1]
        return self

    def predict(self, X):
        n_rows = X.shape[0]
        out = np.zeros((n_rows, self.n_labels_), dtype=int)
        out[:, self.most_frequent_label_idx_] = 1
        return out

    def predict_proba(self, X):
        n_rows = X.shape[0]
        probs = np.tile(self.label_frequencies_, (n_rows, 1))
        return probs


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def normalize_price_string(s: object) -> str:
    if pd.isna(s):
        return "неизвестно"

    s = str(s).strip().lower()
    s = s.replace("\u200b", "")
    s = s.replace("\xa0", " ")
    s = s.replace(",", ";")
    s = s.replace(":", ";")

    parts = [p.strip() for p in s.split(";") if p.strip()]
    parts = list(dict.fromkeys(parts))

    return ";".join(parts) if parts else "неизвестно"


def split_price_labels(s: str):
    parts = [p.strip() for p in str(s).split(";") if p.strip()]
    return parts if parts else ["неизвестно"]


def make_multilabel_target(series: pd.Series):
    normalized = series.apply(normalize_price_string)
    label_lists = normalized.apply(split_price_labels)

    mlb = MultiLabelBinarizer(classes=PRICE_LABEL_ORDER)
    y = mlb.fit_transform(label_lists)

    return normalized, label_lists, mlb, y


def build_preprocessor() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                    ("scaler", StandardScaler()),
                ]),
                NUMERIC_FEATURES,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="constant", fill_value="неизвестно")),
                    ("ohe", make_ohe()),
                ]),
                CATEGORICAL_FEATURES,
            ),
            (
                "desc_tfidf",
                TfidfVectorizer(
                    max_features=4000,
                    ngram_range=(1, 2),
                    min_df=2,
                ),
                "description",
            ),
            (
                "name_tfidf",
                TfidfVectorizer(
                    max_features=1500,
                    ngram_range=(1, 2),
                    min_df=2,
                ),
                "name_clean",
            ),
        ],
        remainder="drop",
    )


def load_and_prepare_data(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = prepare_features(df)

    ensure_required_columns(df, REQUIRED_COLUMNS)

    df["description"] = df["description"].fillna("").astype(str)
    df["name_clean"] = df["name_clean"].fillna("").astype(str)
    df["country_origin"] = df["country_origin"].fillna("Неизвестно").astype(str)
    df["domain"] = df["domain"].fillna("Неизвестно").astype(str)
    df["price_category"] = df["price_category"].fillna("неизвестно").astype(str)

    return df


def build_candidates():
    preprocessor = build_preprocessor()

    candidates = [
        {
            "name": "DummyMostFrequent",
            "family": "baseline",
            "pipeline": Pipeline([
                ("preprocessor", preprocessor),
                ("model", MultiLabelMostFrequentDummy()),
            ]),
            "param_grid": None,
        },
        {
            "name": "LogisticRegression",
            "family": "linear",
            "pipeline": Pipeline([
                ("preprocessor", preprocessor),
                ("model", OneVsRestClassifier(
                    LogisticRegression(
                        max_iter=3000,
                        class_weight="balanced",
                        solver="liblinear",
                        random_state=RANDOM_STATE,
                    )
                )),
            ]),
            "param_grid": {
                "model__estimator__C": [0.3, 1.0, 3.0],
            },
        },
        {
            "name": "RandomForest",
            "family": "bagging",
            "pipeline": Pipeline([
                ("preprocessor", preprocessor),
                ("model", OneVsRestClassifier(
                    RandomForestClassifier(
                        n_estimators=400,
                        class_weight="balanced_subsample",
                        n_jobs=-1,
                        random_state=RANDOM_STATE,
                    )
                )),
            ]),
            "param_grid": {
                "model__estimator__max_depth": [12, 20],
                "model__estimator__min_samples_leaf": [1, 2],
            },
        },
    ]

    if HAS_XGBOOST:
        candidates.append(
            {
                "name": "XGBoost",
                "family": "boosting",
                "pipeline": Pipeline([
                    ("preprocessor", preprocessor),
                    ("model", OneVsRestClassifier(
                        XGBClassifier(
                            n_estimators=300,
                            learning_rate=0.1,
                            max_depth=4,
                            subsample=0.9,
                            colsample_bytree=0.9,
                            reg_lambda=1.0,
                            random_state=RANDOM_STATE,
                            n_jobs=-1,
                            eval_metric="logloss",
                        )
                    )),
                ]),
                "param_grid": {
                    "model__estimator__learning_rate": [0.05, 0.1],
                    "model__estimator__max_depth": [4, 6],
                },
            }
        )

    return candidates


def optimize_thresholds(y_true: np.ndarray, y_prob: np.ndarray, labels: list) -> dict:
    thresholds = {}

    for i, label in enumerate(labels):
        best_thr = 0.5
        best_f1 = -1.0

        for thr in np.arange(0.10, 0.91, 0.05):
            pred = (y_prob[:, i] >= thr).astype(int)
            score = f1_score(y_true[:, i], pred, zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_thr = float(np.round(thr, 2))

        thresholds[label] = best_thr

    return thresholds


def apply_thresholds(y_prob: np.ndarray, thresholds: dict, labels: list) -> np.ndarray:
    out = np.zeros_like(y_prob, dtype=int)

    for i, label in enumerate(labels):
        thr = thresholds[label]
        out[:, i] = (y_prob[:, i] >= thr).astype(int)

    row_sums = out.sum(axis=1)
    empty_rows = np.where(row_sums == 0)[0]

    if len(empty_rows) > 0:
        max_idx = np.argmax(y_prob[empty_rows], axis=1)
        out[empty_rows, max_idx] = 1

    return out


def multilabel_metrics_dict(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "samples_f1": float(f1_score(y_true, y_pred, average="samples", zero_division=0)),
    }


def classification_report_text(y_true: np.ndarray, y_pred: np.ndarray, labels: list) -> str:
    return classification_report(
        y_true,
        y_pred,
        target_names=labels,
        zero_division=0,
    )


def get_probabilities(fitted_model, X) -> np.ndarray:
    """
    ВАЖНО:
    predict_proba вызываем у всего pipeline, а не у внутренней модели,
    чтобы сначала отработал preprocessor.
    """
    if hasattr(fitted_model, "predict_proba"):
        probs = fitted_model.predict_proba(X)

        if isinstance(probs, list):
            cols = []
            for p in probs:
                p = np.asarray(p)
                if p.ndim == 2 and p.shape[1] == 2:
                    cols.append(p[:, 1])
                else:
                    cols.append(p.reshape(-1))
            return np.column_stack(cols)

        if isinstance(probs, np.ndarray):
            if probs.ndim == 3 and probs.shape[2] == 2:
                return probs[:, :, 1]
            return probs

    preds = fitted_model.predict(X)
    return np.asarray(preds, dtype=float)


def cross_validated_threshold_score(candidate, X_train, y_train, labels, n_splits=5):
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    fold_scores = []

    for fold_idx, (tr_idx, va_idx) in enumerate(cv.split(X_train), start=1):
        X_tr = X_train.iloc[tr_idx]
        X_va = X_train.iloc[va_idx]
        y_tr = y_train[tr_idx]
        y_va = y_train[va_idx]

        model = clone(candidate["pipeline"])
        model.fit(X_tr, y_tr)

        y_prob = get_probabilities(model, X_va)
        thresholds = optimize_thresholds(y_va, y_prob, labels)
        y_pred_thr = apply_thresholds(y_prob, thresholds, labels)

        score = f1_score(y_va, y_pred_thr, average="macro", zero_division=0)
        fold_scores.append(float(score))

        print(f"CV fold {fold_idx}: thresholded macro F1 = {score:.4f}")

    return float(np.mean(fold_scores))


def evaluate_candidate(candidate, X_train, y_train, X_test, y_test, labels, inner_cv):
    pipe = clone(candidate["pipeline"])

    if candidate["param_grid"]:
        search = GridSearchCV(
            estimator=pipe,
            param_grid=candidate["param_grid"],
            scoring="f1_macro",
            cv=inner_cv,
            n_jobs=-1,
            verbose=1,
            refit=True,
        )
        search.fit(X_train, y_train)
        fitted_model = search.best_estimator_
        best_params = search.best_params_
        cv_best_macro_f1 = float(search.best_score_)
    else:
        fitted_model = pipe.fit(X_train, y_train)
        best_params = {}
        cv_best_macro_f1 = None

    cv_threshold_macro_f1 = cross_validated_threshold_score(
        candidate=candidate,
        X_train=X_train,
        y_train=y_train,
        labels=labels,
        n_splits=5,
    )

    y_pred_direct = fitted_model.predict(X_test)
    direct_metrics = multilabel_metrics_dict(y_test, y_pred_direct)

    y_prob_train = get_probabilities(fitted_model, X_train)
    thresholds = optimize_thresholds(y_train, y_prob_train, labels)

    y_prob_test = get_probabilities(fitted_model, X_test)
    y_pred_threshold = apply_thresholds(y_prob_test, thresholds, labels)
    threshold_metrics = multilabel_metrics_dict(y_test, y_pred_threshold)

    report_text = classification_report_text(y_test, y_pred_threshold, labels)

    print("\n--- DIRECT PREDICT ---")
    print(f"Macro F1      : {direct_metrics['macro_f1']:.4f}")
    print(f"Micro F1      : {direct_metrics['micro_f1']:.4f}")
    print(f"Weighted F1   : {direct_metrics['weighted_f1']:.4f}")
    print(f"Samples F1    : {direct_metrics['samples_f1']:.4f}")

    print("\n--- THRESHOLDED PREDICT ---")
    print(f"Macro F1      : {threshold_metrics['macro_f1']:.4f}")
    print(f"Micro F1      : {threshold_metrics['micro_f1']:.4f}")
    print(f"Weighted F1   : {threshold_metrics['weighted_f1']:.4f}")
    print(f"Samples F1    : {threshold_metrics['samples_f1']:.4f}")

    print("\n--- CV DIAGNOSTICS ---")
    print(f"GridSearch CV macro F1        : {cv_best_macro_f1}")
    print(f"Manual threshold-CV macro F1  : {cv_threshold_macro_f1:.4f}")

    print("\nUsed thresholds:")
    for label in labels:
        print(f"  {label}: {thresholds[label]:.2f}")

    print("\nClassification report (thresholded predict):")
    print(report_text)

    return {
        "name": candidate["name"],
        "family": candidate["family"],
        "model": fitted_model,
        "best_params": best_params,
        "cv_best_macro_f1": cv_best_macro_f1,
        "cv_threshold_macro_f1": cv_threshold_macro_f1,
        "direct_macro_f1": direct_metrics["macro_f1"],
        "direct_micro_f1": direct_metrics["micro_f1"],
        "threshold_macro_f1": threshold_metrics["macro_f1"],
        "threshold_micro_f1": threshold_metrics["micro_f1"],
        "threshold_weighted_f1": threshold_metrics["weighted_f1"],
        "threshold_samples_f1": threshold_metrics["samples_f1"],
        "thresholds": thresholds,
        "classification_report": report_text,
    }


def save_artifacts(
    out_dir: Path,
    winner: dict,
    leaderboard: pd.DataFrame,
    labels: list,
    rows_used: int,
    feature_columns: list,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    leaderboard.to_csv(out_dir / "leaderboard.csv", index=False, encoding="utf-8-sig")
    joblib.dump(winner["model"], out_dir / "price_category_model.joblib")

    with open(out_dir / "price_category_thresholds.json", "w", encoding="utf-8") as f:
        json.dump(winner["thresholds"], f, ensure_ascii=False, indent=2)

    with open(out_dir / "price_category_labels.json", "w", encoding="utf-8") as f:
        json.dump(labels, f, ensure_ascii=False, indent=2)

    report = {
        "target": "price_category",
        "winner_name": winner["name"],
        "winner_family": winner["family"],
        "rows_used": int(rows_used),
        "feature_columns": feature_columns,
        "labels": labels,
        "best_params": winner["best_params"],
        "cv_best_macro_f1": winner["cv_best_macro_f1"],
        "cv_threshold_macro_f1": winner["cv_threshold_macro_f1"],
        "direct_macro_f1": winner["direct_macro_f1"],
        "direct_micro_f1": winner["direct_micro_f1"],
        "threshold_macro_f1": winner["threshold_macro_f1"],
        "threshold_micro_f1": winner["threshold_micro_f1"],
        "threshold_weighted_f1": winner["threshold_weighted_f1"],
        "threshold_samples_f1": winner["threshold_samples_f1"],
    }

    with open(out_dir / "price_category_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    with open(out_dir / "price_category_classification_report.txt", "w", encoding="utf-8") as f:
        f.write(winner["classification_report"])


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--out_dir",
        type=str,
        default="/kaggle/working/artifacts/price_category",
    )
    parser.add_argument(
        "--test_size",
        type=float,
        default=0.2,
    )
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    df = load_and_prepare_data(args.csv_path)

    _, _, mlb, y = make_multilabel_target(df["price_category"])
    labels = list(mlb.classes_)

    X = df[FEATURE_COLUMNS].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=args.test_size,
        random_state=RANDOM_STATE,
    )

    inner_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    candidates = build_candidates()

    all_results = []

    for candidate in candidates:
        print("\n==============================")
        print(f"Training: {candidate['name']} [{candidate['family']}]")
        print("==============================")

        result = evaluate_candidate(
            candidate=candidate,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            labels=labels,
            inner_cv=inner_cv,
        )
        all_results.append(result)

    leaderboard = pd.DataFrame([
        {
            "name": r["name"],
            "family": r["family"],
            "cv_best_macro_f1": r["cv_best_macro_f1"],
            "cv_threshold_macro_f1": r["cv_threshold_macro_f1"],
            "direct_macro_f1": r["direct_macro_f1"],
            "threshold_macro_f1": r["threshold_macro_f1"],
            "threshold_micro_f1": r["threshold_micro_f1"],
        }
        for r in all_results
    ]).sort_values(
        by=["threshold_macro_f1", "threshold_micro_f1"],
        ascending=[False, False],
    ).reset_index(drop=True)

    winner_name = leaderboard.iloc[0]["name"]
    winner = next(r for r in all_results if r["name"] == winner_name)

    print("\n==============================")
    print("FINAL LEADERBOARD")
    print("==============================")
    print(leaderboard)

    winner_print = {
        "name": winner["name"],
        "family": winner["family"],
        "best_params": winner["best_params"],
        "cv_best_macro_f1": winner["cv_best_macro_f1"],
        "cv_threshold_macro_f1": winner["cv_threshold_macro_f1"],
        "direct_macro_f1": winner["direct_macro_f1"],
        "direct_micro_f1": winner["direct_micro_f1"],
        "threshold_macro_f1": winner["threshold_macro_f1"],
        "threshold_micro_f1": winner["threshold_micro_f1"],
        "threshold_weighted_f1": winner["threshold_weighted_f1"],
        "threshold_samples_f1": winner["threshold_samples_f1"],
    }

    print("\nWinner:")
    print(json.dumps(winner_print, ensure_ascii=False, indent=2))

    save_artifacts(
        out_dir=out_dir,
        winner=winner,
        leaderboard=leaderboard,
        labels=labels,
        rows_used=len(df),
        feature_columns=FEATURE_COLUMNS,
    )

    print(f"\nArtifacts saved to: {out_dir}")


if __name__ == "__main__":
    main()


Training: DummyMostFrequent [baseline]
CV fold 1: thresholded macro F1 = 0.1966
CV fold 2: thresholded macro F1 = 0.2374
CV fold 3: thresholded macro F1 = 0.2075
CV fold 4: thresholded macro F1 = 0.2005
CV fold 5: thresholded macro F1 = 0.2371

--- DIRECT PREDICT ---
Macro F1      : 0.1499
Micro F1      : 0.7356
Weighted F1   : 0.6015
Samples F1    : 0.7603

--- THRESHOLDED PREDICT ---
Macro F1      : 0.2294
Micro F1      : 0.5177
Weighted F1   : 0.6621
Samples F1    : 0.5068

--- CV DIAGNOSTICS ---
GridSearch CV macro F1        : None
Manual threshold-CV macro F1  : 0.2158

Used thresholds:
  дисконт: 0.10
  ниже среднего: 0.10
  средний: 0.10
  выше среднего: 0.10
  люкс / премиум: 0.10
  неизвестно: 0.10

Classification report (thresholded predict):
                precision    recall  f1-score   support

       дисконт       0.00      0.00      0.00        26
 ниже среднего       0.09      1.00      0.16        47
       средний       0.82      1.00      0.90       448
 выше средн

In [34]:
%%writefile qa_checks.py
import argparse
import json
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd

from preprocessing import (
    CANONICAL_COLUMNS,
    harmonize_schema,
    prepare_features,
    ensure_required_columns,
)


REQUIRED_COLUMNS = [
    "name",
    "description",
    "price_category",
    "country_origin",
    "domain",
    "presence_world",
    "presence_russia",
    "presence_regions",
    "plans",
    "founded",
]


def safe_float(x):
    if pd.isna(x):
        return None
    try:
        return float(x)
    except Exception:
        return None


def build_schema_report(df_raw: pd.DataFrame, df: pd.DataFrame) -> Dict:
    raw_cols = list(df_raw.columns)
    prepared_cols = list(df.columns)

    missing_required = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    missing_canonical = [c for c in CANONICAL_COLUMNS if c not in df.columns]
    extra_columns = [c for c in prepared_cols if c not in CANONICAL_COLUMNS]

    return {
        "raw_columns": raw_cols,
        "prepared_columns": prepared_cols,
        "missing_required_columns": missing_required,
        "missing_canonical_columns": missing_canonical,
        "extra_columns_after_preparation": extra_columns,
        "raw_n_columns": len(raw_cols),
        "prepared_n_columns": len(prepared_cols),
    }


def build_missing_report(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    n = len(df)

    for col in df.columns:
        n_missing = int(df[col].isna().sum())
        rows.append({
            "column": col,
            "missing_count": n_missing,
            "missing_rate": float(n_missing / n) if n else 0.0,
            "dtype": str(df[col].dtype),
        })

    return pd.DataFrame(rows).sort_values(
        by=["missing_rate", "missing_count"],
        ascending=[False, False],
    )


def build_text_quality_report(df: pd.DataFrame) -> pd.DataFrame:
    text_cols = [
        c for c in [
            "name",
            "name_clean",
            "description",
            "country_origin",
            "domain",
            "price_category",
        ]
        if c in df.columns
    ]

    rows = []
    for col in text_cols:
        s = df[col].fillna("").astype(str)

        rows.append({
            "column": col,
            "empty_count": int((s.str.strip() == "").sum()),
            "empty_rate": float((s.str.strip() == "").mean()) if len(s) else 0.0,
            "avg_length": float(s.str.len().mean()) if len(s) else 0.0,
            "median_length": float(s.str.len().median()) if len(s) else 0.0,
            "n_unique": int(s.nunique()),
        })

    return pd.DataFrame(rows).sort_values(by="empty_rate", ascending=False)


def build_numeric_summary(df: pd.DataFrame) -> pd.DataFrame:
    numeric_cols = [
        c for c in [
            "founded",
            "presence_world",
            "plans",
            "total_rented_area",
            "presence_own",
            "presence_franchise",
            "n_regions",
            "desc_len",
            "desc_has_digits",
            "desc_year_mentions",
            "has_total_rented_area",
        ]
        if c in df.columns
    ]

    rows = []
    for col in numeric_cols:
        s = pd.to_numeric(df[col], errors="coerce")
        non_null = s.dropna()

        if len(non_null) == 0:
            rows.append({
                "column": col,
                "count_non_null": 0,
                "min": None,
                "p25": None,
                "median": None,
                "p75": None,
                "max": None,
                "mean": None,
            })
            continue

        rows.append({
            "column": col,
            "count_non_null": int(non_null.shape[0]),
            "min": safe_float(non_null.min()),
            "p25": safe_float(non_null.quantile(0.25)),
            "median": safe_float(non_null.median()),
            "p75": safe_float(non_null.quantile(0.75)),
            "max": safe_float(non_null.max()),
            "mean": safe_float(non_null.mean()),
        })

    return pd.DataFrame(rows)


def build_anomaly_report(df: pd.DataFrame) -> Dict:
    report = {}

    if "founded" in df.columns:
        founded = pd.to_numeric(df["founded"], errors="coerce")
        report["founded_lt_1850"] = int((founded < 1850).sum())
        report["founded_gt_2026"] = int((founded > 2026).sum())
        report["founded_missing"] = int(founded.isna().sum())

    if "presence_world" in df.columns:
        s = pd.to_numeric(df["presence_world"], errors="coerce")
        report["presence_world_negative"] = int((s < 0).sum())

    if "plans" in df.columns:
        s = pd.to_numeric(df["plans"], errors="coerce")
        report["plans_negative"] = int((s < 0).sum())

    if "total_rented_area" in df.columns:
        s = pd.to_numeric(df["total_rented_area"], errors="coerce")
        report["total_rented_area_negative"] = int((s < 0).sum())
        report["total_rented_area_known_count"] = int(s.notna().sum())
        report["total_rented_area_known_rate"] = float(s.notna().mean()) if len(df) else 0.0

    if "description" in df.columns:
        desc = df["description"].fillna("").astype(str)
        report["description_empty"] = int((desc.str.strip() == "").sum())

    return report


def build_duplicate_report(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    out = {}

    if "name" in df.columns:
        exact_name_dupes = (
            df.groupby("name")
            .size()
            .reset_index(name="rows")
            .sort_values(by="rows", ascending=False)
        )
        out["exact_name_duplicates"] = exact_name_dupes[exact_name_dupes["rows"] > 1].copy()

    if "name_clean" in df.columns:
        clean_name_dupes = (
            df.groupby("name_clean")
            .size()
            .reset_index(name="rows")
            .sort_values(by="rows", ascending=False)
        )
        out["clean_name_duplicates"] = clean_name_dupes[clean_name_dupes["rows"] > 1].copy()

        if "name" in df.columns:
            ambiguous = (
                df.groupby("name_clean")["name"]
                .nunique()
                .reset_index(name="n_original_name_variants")
                .sort_values(by="n_original_name_variants", ascending=False)
            )
            out["ambiguous_clean_names"] = ambiguous[ambiguous["n_original_name_variants"] > 1].copy()

    return out


def build_target_reports(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    reports = {}

    if "domain" in df.columns:
        domain_counts = (
            df["domain"]
            .fillna("MISSING")
            .astype(str)
            .value_counts(dropna=False)
            .rename_axis("domain")
            .reset_index(name="count")
        )
        domain_counts["rate"] = domain_counts["count"] / len(df) if len(df) else 0.0
        reports["domain_distribution"] = domain_counts

    if "price_category" in df.columns:
        price_counts = (
            df["price_category"]
            .fillna("MISSING")
            .astype(str)
            .value_counts(dropna=False)
            .rename_axis("price_category")
            .reset_index(name="count")
        )
        price_counts["rate"] = price_counts["count"] / len(df) if len(df) else 0.0
        reports["price_category_distribution"] = price_counts

        token_rows = []
        for value in df["price_category"].fillna("неизвестно").astype(str):
            parts = [p.strip() for p in value.split(";") if p.strip()]
            token_rows.extend(parts)

        if token_rows:
            token_counts = (
                pd.Series(token_rows)
                .value_counts()
                .rename_axis("label")
                .reset_index(name="count")
            )
            token_counts["rate"] = token_counts["count"] / len(df) if len(df) else 0.0
        else:
            token_counts = pd.DataFrame(columns=["label", "count", "rate"])

        reports["price_category_label_distribution"] = token_counts

    if "founded" in df.columns:
        founded = pd.to_numeric(df["founded"], errors="coerce")
        decade = (np.floor(founded / 10) * 10).astype("Int64")
        decade_counts = (
            decade.dropna()
            .astype(int)
            .value_counts()
            .sort_index()
            .rename_axis("decade")
            .reset_index(name="count")
        )
        decade_counts["rate"] = decade_counts["count"] / max(1, founded.notna().sum())
        reports["founded_decade_distribution"] = decade_counts

    return reports


def build_conflict_report(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    reports = {}

    key_cols = [c for c in ["name", "country_origin", "domain", "price_category"] if c in df.columns]

    if len(key_cols) == 4 and "founded" in df.columns:
        founded_conflicts = (
            df.groupby(key_cols)["founded"]
            .nunique(dropna=True)
            .reset_index(name="n_founded_variants")
        )
        founded_conflicts = founded_conflicts[founded_conflicts["n_founded_variants"] > 1].copy()
        reports["founded_conflicts"] = founded_conflicts.sort_values(
            by="n_founded_variants", ascending=False
        )

    if "name_clean" in df.columns and "domain" in df.columns:
        domain_conflicts = (
            df.groupby("name_clean")["domain"]
            .nunique(dropna=True)
            .reset_index(name="n_domain_variants")
        )
        domain_conflicts = domain_conflicts[domain_conflicts["n_domain_variants"] > 1].copy()
        reports["name_clean_domain_conflicts"] = domain_conflicts.sort_values(
            by="n_domain_variants", ascending=False
        )

    if "name_clean" in df.columns and "country_origin" in df.columns:
        country_conflicts = (
            df.groupby("name_clean")["country_origin"]
            .nunique(dropna=True)
            .reset_index(name="n_country_variants")
        )
        country_conflicts = country_conflicts[country_conflicts["n_country_variants"] > 1].copy()
        reports["name_clean_country_conflicts"] = country_conflicts.sort_values(
            by="n_country_variants", ascending=False
        )

    return reports


def build_training_readiness_report(df: pd.DataFrame) -> Dict:
    report = {"rows_total": int(len(df))}

    if "domain" in df.columns:
        tmp = df[df["domain"].notna()].copy()
        tmp["domain"] = tmp["domain"].astype(str).str.strip()
        value_counts = tmp["domain"].value_counts()
        keep = value_counts[value_counts >= 5]
        report["domain_task"] = {
            "rows_non_null_target": int(tmp.shape[0]),
            "n_classes_all": int(value_counts.shape[0]),
            "n_classes_ge_5": int(keep.shape[0]),
            "rows_after_drop_rare_classes": int(tmp[tmp["domain"].isin(keep.index)].shape[0]),
        }

    if "founded" in df.columns:
        founded = pd.to_numeric(df["founded"], errors="coerce")
        valid = founded.between(1850, 2025)
        report["founded_task"] = {
            "rows_non_null_target": int(founded.notna().sum()),
            "rows_in_valid_range_1850_2025": int(valid.sum()),
        }

    if "price_category" in df.columns:
        non_null = df["price_category"].fillna("неизвестно").astype(str)
        report["price_category_task"] = {
            "rows_non_null_target": int((non_null != "").sum()),
            "n_unique_combinations": int(non_null.nunique()),
        }

    return report


def save_dataframe(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")


def main():
    parser = argparse.ArgumentParser(description="QA checks for retail enrichment project.")
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--out_dir",
        type=str,
        default="/kaggle/working/artifacts/qa",
    )
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print("Loading dataset...")
    df_raw = pd.read_csv(args.csv_path)

    print("Harmonizing schema...")
    df_harmonized = harmonize_schema(df_raw)

    print("Running feature preparation...")
    df = prepare_features(df_harmonized)

    print("Checking required columns...")
    missing_required = [c for c in REQUIRED_COLUMNS if c not in df.columns]

    schema_report = build_schema_report(df_raw, df)
    missing_report = build_missing_report(df)
    text_quality_report = build_text_quality_report(df)
    numeric_summary = build_numeric_summary(df)
    anomaly_report = build_anomaly_report(df)
    duplicate_reports = build_duplicate_report(df)
    target_reports = build_target_reports(df)
    conflict_reports = build_conflict_report(df)
    training_readiness = build_training_readiness_report(df)

    summary = {
        "dataset_shape_raw": [int(df_raw.shape[0]), int(df_raw.shape[1])],
        "dataset_shape_prepared": [int(df.shape[0]), int(df.shape[1])],
        "schema_report": schema_report,
        "anomaly_report": anomaly_report,
        "training_readiness": training_readiness,
    }

    if missing_required:
        summary["required_columns_check"] = {
            "status": "FAILED",
            "missing_required_columns": missing_required,
        }
    else:
        summary["required_columns_check"] = {
            "status": "OK",
            "missing_required_columns": [],
        }

    print("Saving reports...")
    (out_dir / "qa_summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    save_dataframe(missing_report, out_dir / "missing_report.csv")
    save_dataframe(text_quality_report, out_dir / "text_quality_report.csv")
    save_dataframe(numeric_summary, out_dir / "numeric_summary.csv")

    for name, rep in duplicate_reports.items():
        save_dataframe(rep, out_dir / f"{name}.csv")

    for name, rep in target_reports.items():
        save_dataframe(rep, out_dir / f"{name}.csv")

    for name, rep in conflict_reports.items():
        save_dataframe(rep, out_dir / f"{name}.csv")

    preview_cols = [c for c in [
        "name",
        "name_clean",
        "country_origin",
        "domain",
        "price_category",
        "founded",
        "presence_world",
        "presence_russia",
        "presence_own",
        "presence_franchise",
        "n_regions",
        "desc_len",
        "desc_has_digits",
        "desc_year_mentions",
        "has_total_rented_area",
    ] if c in df.columns]

    save_dataframe(df[preview_cols].head(1000), out_dir / "prepared_preview_1000.csv")

    print("\n==============================")
    print("QA CHECKS COMPLETE")
    print("==============================")
    print(f"Rows raw              : {df_raw.shape[0]}")
    print(f"Cols raw              : {df_raw.shape[1]}")
    print(f"Rows prepared         : {df.shape[0]}")
    print(f"Cols prepared         : {df.shape[1]}")
    print(f"Required columns OK   : {not bool(missing_required)}")

    if "domain_task" in training_readiness:
        print("\n[domain]")
        print(f"Rows with target      : {training_readiness['domain_task']['rows_non_null_target']}")
        print(f"Classes total         : {training_readiness['domain_task']['n_classes_all']}")
        print(f"Classes >= 5 rows     : {training_readiness['domain_task']['n_classes_ge_5']}")
        print(f"Rows after class cut  : {training_readiness['domain_task']['rows_after_drop_rare_classes']}")

    if "founded_task" in training_readiness:
        print("\n[founded]")
        print(f"Rows with target      : {training_readiness['founded_task']['rows_non_null_target']}")
        print(f"Rows in valid range   : {training_readiness['founded_task']['rows_in_valid_range_1850_2025']}")

    if "price_category_task" in training_readiness:
        print("\n[price_category]")
        print(f"Rows with target      : {training_readiness['price_category_task']['rows_non_null_target']}")
        print(f"Unique combinations   : {training_readiness['price_category_task']['n_unique_combinations']}")

    if "total_rented_area_known_rate" in anomaly_report:
        print("\n[total_rented_area]")
        print(f"Known count           : {anomaly_report['total_rented_area_known_count']}")
        print(f"Known rate            : {anomaly_report['total_rented_area_known_rate']:.4f}")

    print(f"\nArtifacts saved to: {out_dir}")


if __name__ == "__main__":
    main()

Writing qa_checks.py


In [35]:
import argparse
import json
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd

from preprocessing import (
    CANONICAL_COLUMNS,
    harmonize_schema,
    prepare_features,
    ensure_required_columns,
)


REQUIRED_COLUMNS = [
    "name",
    "description",
    "price_category",
    "country_origin",
    "domain",
    "presence_world",
    "presence_russia",
    "presence_regions",
    "plans",
    "founded",
]


def safe_float(x):
    if pd.isna(x):
        return None
    try:
        return float(x)
    except Exception:
        return None


def build_schema_report(df_raw: pd.DataFrame, df: pd.DataFrame) -> Dict:
    raw_cols = list(df_raw.columns)
    prepared_cols = list(df.columns)

    missing_required = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    missing_canonical = [c for c in CANONICAL_COLUMNS if c not in df.columns]
    extra_columns = [c for c in prepared_cols if c not in CANONICAL_COLUMNS]

    return {
        "raw_columns": raw_cols,
        "prepared_columns": prepared_cols,
        "missing_required_columns": missing_required,
        "missing_canonical_columns": missing_canonical,
        "extra_columns_after_preparation": extra_columns,
        "raw_n_columns": len(raw_cols),
        "prepared_n_columns": len(prepared_cols),
    }


def build_missing_report(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    n = len(df)

    for col in df.columns:
        n_missing = int(df[col].isna().sum())
        rows.append({
            "column": col,
            "missing_count": n_missing,
            "missing_rate": float(n_missing / n) if n else 0.0,
            "dtype": str(df[col].dtype),
        })

    return pd.DataFrame(rows).sort_values(
        by=["missing_rate", "missing_count"],
        ascending=[False, False],
    )


def build_text_quality_report(df: pd.DataFrame) -> pd.DataFrame:
    text_cols = [
        c for c in [
            "name",
            "name_clean",
            "description",
            "country_origin",
            "domain",
            "price_category",
        ]
        if c in df.columns
    ]

    rows = []
    for col in text_cols:
        s = df[col].fillna("").astype(str)

        rows.append({
            "column": col,
            "empty_count": int((s.str.strip() == "").sum()),
            "empty_rate": float((s.str.strip() == "").mean()) if len(s) else 0.0,
            "avg_length": float(s.str.len().mean()) if len(s) else 0.0,
            "median_length": float(s.str.len().median()) if len(s) else 0.0,
            "n_unique": int(s.nunique()),
        })

    return pd.DataFrame(rows).sort_values(by="empty_rate", ascending=False)


def build_numeric_summary(df: pd.DataFrame) -> pd.DataFrame:
    numeric_cols = [
        c for c in [
            "founded",
            "presence_world",
            "plans",
            "total_rented_area",
            "presence_own",
            "presence_franchise",
            "n_regions",
            "desc_len",
            "desc_has_digits",
            "desc_year_mentions",
            "has_total_rented_area",
        ]
        if c in df.columns
    ]

    rows = []
    for col in numeric_cols:
        s = pd.to_numeric(df[col], errors="coerce")
        non_null = s.dropna()

        if len(non_null) == 0:
            rows.append({
                "column": col,
                "count_non_null": 0,
                "min": None,
                "p25": None,
                "median": None,
                "p75": None,
                "max": None,
                "mean": None,
            })
            continue

        rows.append({
            "column": col,
            "count_non_null": int(non_null.shape[0]),
            "min": safe_float(non_null.min()),
            "p25": safe_float(non_null.quantile(0.25)),
            "median": safe_float(non_null.median()),
            "p75": safe_float(non_null.quantile(0.75)),
            "max": safe_float(non_null.max()),
            "mean": safe_float(non_null.mean()),
        })

    return pd.DataFrame(rows)


def build_anomaly_report(df: pd.DataFrame) -> Dict:
    report = {}

    if "founded" in df.columns:
        founded = pd.to_numeric(df["founded"], errors="coerce")
        report["founded_lt_1850"] = int((founded < 1850).sum())
        report["founded_gt_2026"] = int((founded > 2026).sum())
        report["founded_missing"] = int(founded.isna().sum())

    if "presence_world" in df.columns:
        s = pd.to_numeric(df["presence_world"], errors="coerce")
        report["presence_world_negative"] = int((s < 0).sum())

    if "plans" in df.columns:
        s = pd.to_numeric(df["plans"], errors="coerce")
        report["plans_negative"] = int((s < 0).sum())

    if "total_rented_area" in df.columns:
        s = pd.to_numeric(df["total_rented_area"], errors="coerce")
        report["total_rented_area_negative"] = int((s < 0).sum())
        report["total_rented_area_known_count"] = int(s.notna().sum())
        report["total_rented_area_known_rate"] = float(s.notna().mean()) if len(df) else 0.0

    if "description" in df.columns:
        desc = df["description"].fillna("").astype(str)
        report["description_empty"] = int((desc.str.strip() == "").sum())

    return report


def build_duplicate_report(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    out = {}

    if "name" in df.columns:
        exact_name_dupes = (
            df.groupby("name")
            .size()
            .reset_index(name="rows")
            .sort_values(by="rows", ascending=False)
        )
        out["exact_name_duplicates"] = exact_name_dupes[exact_name_dupes["rows"] > 1].copy()

    if "name_clean" in df.columns:
        clean_name_dupes = (
            df.groupby("name_clean")
            .size()
            .reset_index(name="rows")
            .sort_values(by="rows", ascending=False)
        )
        out["clean_name_duplicates"] = clean_name_dupes[clean_name_dupes["rows"] > 1].copy()

        if "name" in df.columns:
            ambiguous = (
                df.groupby("name_clean")["name"]
                .nunique()
                .reset_index(name="n_original_name_variants")
                .sort_values(by="n_original_name_variants", ascending=False)
            )
            out["ambiguous_clean_names"] = ambiguous[ambiguous["n_original_name_variants"] > 1].copy()

    return out


def build_target_reports(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    reports = {}

    if "domain" in df.columns:
        domain_counts = (
            df["domain"]
            .fillna("MISSING")
            .astype(str)
            .value_counts(dropna=False)
            .rename_axis("domain")
            .reset_index(name="count")
        )
        domain_counts["rate"] = domain_counts["count"] / len(df) if len(df) else 0.0
        reports["domain_distribution"] = domain_counts

    if "price_category" in df.columns:
        price_counts = (
            df["price_category"]
            .fillna("MISSING")
            .astype(str)
            .value_counts(dropna=False)
            .rename_axis("price_category")
            .reset_index(name="count")
        )
        price_counts["rate"] = price_counts["count"] / len(df) if len(df) else 0.0
        reports["price_category_distribution"] = price_counts

        token_rows = []
        for value in df["price_category"].fillna("неизвестно").astype(str):
            parts = [p.strip() for p in value.split(";") if p.strip()]
            token_rows.extend(parts)

        if token_rows:
            token_counts = (
                pd.Series(token_rows)
                .value_counts()
                .rename_axis("label")
                .reset_index(name="count")
            )
            token_counts["rate"] = token_counts["count"] / len(df) if len(df) else 0.0
        else:
            token_counts = pd.DataFrame(columns=["label", "count", "rate"])

        reports["price_category_label_distribution"] = token_counts

    if "founded" in df.columns:
        founded = pd.to_numeric(df["founded"], errors="coerce")
        decade = (np.floor(founded / 10) * 10).astype("Int64")
        decade_counts = (
            decade.dropna()
            .astype(int)
            .value_counts()
            .sort_index()
            .rename_axis("decade")
            .reset_index(name="count")
        )
        decade_counts["rate"] = decade_counts["count"] / max(1, founded.notna().sum())
        reports["founded_decade_distribution"] = decade_counts

    return reports


def build_conflict_report(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    reports = {}

    key_cols = [c for c in ["name", "country_origin", "domain", "price_category"] if c in df.columns]

    if len(key_cols) == 4 and "founded" in df.columns:
        founded_conflicts = (
            df.groupby(key_cols)["founded"]
            .nunique(dropna=True)
            .reset_index(name="n_founded_variants")
        )
        founded_conflicts = founded_conflicts[founded_conflicts["n_founded_variants"] > 1].copy()
        reports["founded_conflicts"] = founded_conflicts.sort_values(
            by="n_founded_variants", ascending=False
        )

    if "name_clean" in df.columns and "domain" in df.columns:
        domain_conflicts = (
            df.groupby("name_clean")["domain"]
            .nunique(dropna=True)
            .reset_index(name="n_domain_variants")
        )
        domain_conflicts = domain_conflicts[domain_conflicts["n_domain_variants"] > 1].copy()
        reports["name_clean_domain_conflicts"] = domain_conflicts.sort_values(
            by="n_domain_variants", ascending=False
        )

    if "name_clean" in df.columns and "country_origin" in df.columns:
        country_conflicts = (
            df.groupby("name_clean")["country_origin"]
            .nunique(dropna=True)
            .reset_index(name="n_country_variants")
        )
        country_conflicts = country_conflicts[country_conflicts["n_country_variants"] > 1].copy()
        reports["name_clean_country_conflicts"] = country_conflicts.sort_values(
            by="n_country_variants", ascending=False
        )

    return reports


def build_training_readiness_report(df: pd.DataFrame) -> Dict:
    report = {"rows_total": int(len(df))}

    if "domain" in df.columns:
        tmp = df[df["domain"].notna()].copy()
        tmp["domain"] = tmp["domain"].astype(str).str.strip()
        value_counts = tmp["domain"].value_counts()
        keep = value_counts[value_counts >= 5]
        report["domain_task"] = {
            "rows_non_null_target": int(tmp.shape[0]),
            "n_classes_all": int(value_counts.shape[0]),
            "n_classes_ge_5": int(keep.shape[0]),
            "rows_after_drop_rare_classes": int(tmp[tmp["domain"].isin(keep.index)].shape[0]),
        }

    if "founded" in df.columns:
        founded = pd.to_numeric(df["founded"], errors="coerce")
        valid = founded.between(1850, 2025)
        report["founded_task"] = {
            "rows_non_null_target": int(founded.notna().sum()),
            "rows_in_valid_range_1850_2025": int(valid.sum()),
        }

    if "price_category" in df.columns:
        non_null = df["price_category"].fillna("неизвестно").astype(str)
        report["price_category_task"] = {
            "rows_non_null_target": int((non_null != "").sum()),
            "n_unique_combinations": int(non_null.nunique()),
        }

    return report


def save_dataframe(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")


def main():
    parser = argparse.ArgumentParser(description="QA checks for retail enrichment project.")
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--out_dir",
        type=str,
        default="/kaggle/working/artifacts/qa",
    )
    args, _ = parser.parse_known_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print("Loading dataset...")
    df_raw = pd.read_csv(args.csv_path)

    print("Harmonizing schema...")
    df_harmonized = harmonize_schema(df_raw)

    print("Running feature preparation...")
    df = prepare_features(df_harmonized)

    print("Checking required columns...")
    missing_required = [c for c in REQUIRED_COLUMNS if c not in df.columns]

    schema_report = build_schema_report(df_raw, df)
    missing_report = build_missing_report(df)
    text_quality_report = build_text_quality_report(df)
    numeric_summary = build_numeric_summary(df)
    anomaly_report = build_anomaly_report(df)
    duplicate_reports = build_duplicate_report(df)
    target_reports = build_target_reports(df)
    conflict_reports = build_conflict_report(df)
    training_readiness = build_training_readiness_report(df)

    summary = {
        "dataset_shape_raw": [int(df_raw.shape[0]), int(df_raw.shape[1])],
        "dataset_shape_prepared": [int(df.shape[0]), int(df.shape[1])],
        "schema_report": schema_report,
        "anomaly_report": anomaly_report,
        "training_readiness": training_readiness,
    }

    if missing_required:
        summary["required_columns_check"] = {
            "status": "FAILED",
            "missing_required_columns": missing_required,
        }
    else:
        summary["required_columns_check"] = {
            "status": "OK",
            "missing_required_columns": [],
        }

    print("Saving reports...")
    (out_dir / "qa_summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    save_dataframe(missing_report, out_dir / "missing_report.csv")
    save_dataframe(text_quality_report, out_dir / "text_quality_report.csv")
    save_dataframe(numeric_summary, out_dir / "numeric_summary.csv")

    for name, rep in duplicate_reports.items():
        save_dataframe(rep, out_dir / f"{name}.csv")

    for name, rep in target_reports.items():
        save_dataframe(rep, out_dir / f"{name}.csv")

    for name, rep in conflict_reports.items():
        save_dataframe(rep, out_dir / f"{name}.csv")

    preview_cols = [c for c in [
        "name",
        "name_clean",
        "country_origin",
        "domain",
        "price_category",
        "founded",
        "presence_world",
        "presence_russia",
        "presence_own",
        "presence_franchise",
        "n_regions",
        "desc_len",
        "desc_has_digits",
        "desc_year_mentions",
        "has_total_rented_area",
    ] if c in df.columns]

    save_dataframe(df[preview_cols].head(1000), out_dir / "prepared_preview_1000.csv")

    print("\n==============================")
    print("QA CHECKS COMPLETE")
    print("==============================")
    print(f"Rows raw              : {df_raw.shape[0]}")
    print(f"Cols raw              : {df_raw.shape[1]}")
    print(f"Rows prepared         : {df.shape[0]}")
    print(f"Cols prepared         : {df.shape[1]}")
    print(f"Required columns OK   : {not bool(missing_required)}")

    if "domain_task" in training_readiness:
        print("\n[domain]")
        print(f"Rows with target      : {training_readiness['domain_task']['rows_non_null_target']}")
        print(f"Classes total         : {training_readiness['domain_task']['n_classes_all']}")
        print(f"Classes >= 5 rows     : {training_readiness['domain_task']['n_classes_ge_5']}")
        print(f"Rows after class cut  : {training_readiness['domain_task']['rows_after_drop_rare_classes']}")

    if "founded_task" in training_readiness:
        print("\n[founded]")
        print(f"Rows with target      : {training_readiness['founded_task']['rows_non_null_target']}")
        print(f"Rows in valid range   : {training_readiness['founded_task']['rows_in_valid_range_1850_2025']}")

    if "price_category_task" in training_readiness:
        print("\n[price_category]")
        print(f"Rows with target      : {training_readiness['price_category_task']['rows_non_null_target']}")
        print(f"Unique combinations   : {training_readiness['price_category_task']['n_unique_combinations']}")

    if "total_rented_area_known_rate" in anomaly_report:
        print("\n[total_rented_area]")
        print(f"Known count           : {anomaly_report['total_rented_area_known_count']}")
        print(f"Known rate            : {anomaly_report['total_rented_area_known_rate']:.4f}")

    print(f"\nArtifacts saved to: {out_dir}")


if __name__ == "__main__":
    main()

Loading dataset...
Harmonizing schema...
Running feature preparation...
Checking required columns...
Saving reports...

QA CHECKS COMPLETE
Rows raw              : 2737
Cols raw              : 11
Rows prepared         : 2737
Cols prepared         : 19
Required columns OK   : True

[domain]
Rows with target      : 2737
Classes total         : 40
Classes >= 5 rows     : 38
Rows after class cut  : 2733

[founded]
Rows with target      : 2533
Rows in valid range   : 2522

[price_category]
Rows with target      : 2737
Unique combinations   : 19

[total_rented_area]
Known count           : 117
Known rate            : 0.0427

Artifacts saved to: /kaggle/working/artifacts/qa


In [39]:
%%writefile predict_all.py
import argparse
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from preprocessing import prepare_features


DOMAIN_FEATURE_COLUMNS = [
    "description",
    "name_clean",
    "price_category",
    "country_origin",
    "founded",
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]

FOUNDED_FEATURE_COLUMNS = [
    "description",
    "name_clean",
    "price_category",
    "country_origin",
    "domain",
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]

PRICE_FEATURE_COLUMNS = [
    "description",
    "name_clean",
    "country_origin",
    "domain",
    "founded",
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]

TEXT_COLUMNS = [
    "description",
    "name_clean",
]

CATEGORICAL_COLUMNS = [
    "country_origin",
    "domain",
    "price_category",
]

NUMERIC_COLUMNS = [
    "founded",
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


def ensure_columns(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    df = df.copy()

    for col in columns:
        if col not in df.columns:
            if col in TEXT_COLUMNS:
                df[col] = ""
            elif col in CATEGORICAL_COLUMNS:
                if col == "country_origin":
                    df[col] = "Неизвестно"
                elif col == "price_category":
                    df[col] = "неизвестно"
                else:
                    df[col] = "Неизвестно"
            else:
                df[col] = np.nan

    return df


def sanitize_for_model(df: pd.DataFrame, feature_columns: list) -> pd.DataFrame:
    df = ensure_columns(df, feature_columns).copy()

    for col in TEXT_COLUMNS:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str)

    for col in CATEGORICAL_COLUMNS:
        if col in df.columns:
            if col == "country_origin":
                df[col] = df[col].fillna("Неизвестно").astype(str)
            elif col == "price_category":
                df[col] = df[col].fillna("неизвестно").astype(str)
            else:
                df[col] = df[col].fillna("Неизвестно").astype(str)

    for col in NUMERIC_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df[feature_columns].copy()


def get_price_probabilities(fitted_model, X) -> np.ndarray:
    if hasattr(fitted_model, "predict_proba"):
        probs = fitted_model.predict_proba(X)

        if isinstance(probs, list):
            cols = []
            for p in probs:
                p = np.asarray(p)
                if p.ndim == 2 and p.shape[1] == 2:
                    cols.append(p[:, 1])
                else:
                    cols.append(p.reshape(-1))
            return np.column_stack(cols)

        if isinstance(probs, np.ndarray):
            if probs.ndim == 3 and probs.shape[2] == 2:
                return probs[:, :, 1]
            return probs

    preds = fitted_model.predict(X)
    return np.asarray(preds, dtype=float)


def apply_thresholds(y_prob: np.ndarray, thresholds: dict, labels: list) -> np.ndarray:
    out = np.zeros_like(y_prob, dtype=int)

    for i, label in enumerate(labels):
        thr = float(thresholds[label])
        out[:, i] = (y_prob[:, i] >= thr).astype(int)

    row_sums = out.sum(axis=1)
    empty_rows = np.where(row_sums == 0)[0]

    if len(empty_rows) > 0:
        max_idx = np.argmax(y_prob[empty_rows], axis=1)
        out[empty_rows, max_idx] = 1

    return out


def multilabel_rows_to_strings(y_bin: np.ndarray, labels: list) -> list:
    results = []
    for row in y_bin:
        active = [label for label, val in zip(labels, row) if val == 1]
        results.append("; ".join(active) if active else "неизвестно")
    return results


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--artifacts_dir",
        type=str,
        default="/kaggle/working/artifacts",
    )
    parser.add_argument(
        "--out_path",
        type=str,
        default="/kaggle/working/artifacts/predictions_all.csv",
    )
    args, _ = parser.parse_known_args()

    artifacts_dir = Path(args.artifacts_dir)
    out_path = Path(args.out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    print("Loading raw data...")
    df_raw = pd.read_csv(args.csv_path)

    print("Preparing features...")
    df = prepare_features(df_raw)

    result = df_raw.copy()

    # ---------- DOMAIN ----------
    domain_model_path = artifacts_dir / "domain" / "domain_model.joblib"
    domain_encoder_path = artifacts_dir / "domain" / "domain_label_encoder.joblib"

    if domain_model_path.exists():
        print("Loading domain model...")
        domain_model = joblib.load(domain_model_path)

        X_domain = sanitize_for_model(df, DOMAIN_FEATURE_COLUMNS)
        domain_pred = domain_model.predict(X_domain)

        if np.issubdtype(np.asarray(domain_pred).dtype, np.integer) and domain_encoder_path.exists():
            domain_encoder = joblib.load(domain_encoder_path)
            domain_pred = domain_encoder.inverse_transform(domain_pred)

        result["pred_domain"] = domain_pred
    else:
        print("Domain model not found, skipping.")
        result["pred_domain"] = np.nan

    # ---------- FOUNDED ----------
    founded_model_path = artifacts_dir / "founded" / "founded_model.joblib"

    if founded_model_path.exists():
        print("Loading founded model...")
        founded_model = joblib.load(founded_model_path)

        founded_df = df.copy()
        if "pred_domain" in result.columns:
            founded_df["domain"] = result["pred_domain"].fillna(founded_df.get("domain", "Неизвестно"))

        X_founded = sanitize_for_model(founded_df, FOUNDED_FEATURE_COLUMNS)
        founded_pred = founded_model.predict(X_founded)
        founded_pred = np.clip(founded_pred, 0, None)

        result["pred_founded"] = founded_pred
        result["pred_founded_rounded"] = np.round(founded_pred).astype(int)
    else:
        print("Founded model not found, skipping.")
        result["pred_founded"] = np.nan
        result["pred_founded_rounded"] = np.nan

    # ---------- PRICE CATEGORY ----------
    price_model_path = artifacts_dir / "price_category" / "price_category_model.joblib"
    price_thresholds_path = artifacts_dir / "price_category" / "price_category_thresholds.json"
    price_labels_path = artifacts_dir / "price_category" / "price_category_labels.json"

    if price_model_path.exists() and price_thresholds_path.exists() and price_labels_path.exists():
        print("Loading price_category model...")
        price_model = joblib.load(price_model_path)

        with open(price_thresholds_path, "r", encoding="utf-8") as f:
            thresholds = json.load(f)

        with open(price_labels_path, "r", encoding="utf-8") as f:
            labels = json.load(f)

        price_df = df.copy()

        if "pred_domain" in result.columns:
            price_df["domain"] = result["pred_domain"].fillna(price_df.get("domain", "Неизвестно"))

        if "pred_founded" in result.columns:
            current_founded = price_df.get("founded", pd.Series(np.nan, index=price_df.index))
            price_df["founded"] = pd.Series(result["pred_founded"], index=price_df.index).fillna(current_founded)

        X_price = sanitize_for_model(price_df, PRICE_FEATURE_COLUMNS)

        y_prob = get_price_probabilities(price_model, X_price)
        y_pred_bin = apply_thresholds(y_prob, thresholds, labels)
        y_pred_text = multilabel_rows_to_strings(y_pred_bin, labels)

        result["pred_price_category"] = y_pred_text

        for i, label in enumerate(labels):
            safe_label = (
                label.replace(" / ", "_")
                .replace(" ", "_")
                .replace("-", "_")
            )
            result[f"pred_price_prob__{safe_label}"] = y_prob[:, i]
            result[f"pred_price_label__{safe_label}"] = y_pred_bin[:, i]
    else:
        print("Price category artifacts not found, skipping.")
        result["pred_price_category"] = np.nan

    print("Saving predictions...")
    result.to_csv(out_path, index=False, encoding="utf-8-sig")

    summary = {
        "rows_scored": int(len(result)),
        "domain_model_used": bool(domain_model_path.exists()),
        "founded_model_used": bool(founded_model_path.exists()),
        "price_category_model_used": bool(price_model_path.exists()),
        "output_path": str(out_path),
    }

    with open(out_path.with_suffix(".json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("\n==============================")
    print("PREDICT ALL COMPLETE")
    print("==============================")
    print(f"Rows scored      : {len(result)}")
    print(f"Saved to         : {out_path}")


if __name__ == "__main__":
    main()

Overwriting predict_all.py


In [40]:
import argparse
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from preprocessing import prepare_features


DOMAIN_FEATURE_COLUMNS = [
    "description",
    "name_clean",
    "price_category",
    "country_origin",
    "founded",
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]

FOUNDED_FEATURE_COLUMNS = [
    "description",
    "name_clean",
    "price_category",
    "country_origin",
    "domain",
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]

PRICE_FEATURE_COLUMNS = [
    "description",
    "name_clean",
    "country_origin",
    "domain",
    "founded",
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]

TEXT_COLUMNS = [
    "description",
    "name_clean",
]

CATEGORICAL_COLUMNS = [
    "country_origin",
    "domain",
    "price_category",
]

NUMERIC_COLUMNS = [
    "founded",
    "presence_world",
    "plans",
    "presence_own",
    "presence_franchise",
    "n_regions",
    "desc_len",
    "desc_has_digits",
    "desc_year_mentions",
    "has_total_rented_area",
]


def ensure_columns(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    df = df.copy()

    for col in columns:
        if col not in df.columns:
            if col in TEXT_COLUMNS:
                df[col] = ""
            elif col in CATEGORICAL_COLUMNS:
                if col == "country_origin":
                    df[col] = "Неизвестно"
                elif col == "price_category":
                    df[col] = "неизвестно"
                else:
                    df[col] = "Неизвестно"
            else:
                df[col] = np.nan

    return df


def sanitize_for_model(df: pd.DataFrame, feature_columns: list) -> pd.DataFrame:
    df = ensure_columns(df, feature_columns).copy()

    for col in TEXT_COLUMNS:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str)

    for col in CATEGORICAL_COLUMNS:
        if col in df.columns:
            if col == "country_origin":
                df[col] = df[col].fillna("Неизвестно").astype(str)
            elif col == "price_category":
                df[col] = df[col].fillna("неизвестно").astype(str)
            else:
                df[col] = df[col].fillna("Неизвестно").astype(str)

    for col in NUMERIC_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df[feature_columns].copy()


def get_price_probabilities(fitted_model, X) -> np.ndarray:
    if hasattr(fitted_model, "predict_proba"):
        probs = fitted_model.predict_proba(X)

        if isinstance(probs, list):
            cols = []
            for p in probs:
                p = np.asarray(p)
                if p.ndim == 2 and p.shape[1] == 2:
                    cols.append(p[:, 1])
                else:
                    cols.append(p.reshape(-1))
            return np.column_stack(cols)

        if isinstance(probs, np.ndarray):
            if probs.ndim == 3 and probs.shape[2] == 2:
                return probs[:, :, 1]
            return probs

    preds = fitted_model.predict(X)
    return np.asarray(preds, dtype=float)


def apply_thresholds(y_prob: np.ndarray, thresholds: dict, labels: list) -> np.ndarray:
    out = np.zeros_like(y_prob, dtype=int)

    for i, label in enumerate(labels):
        thr = float(thresholds[label])
        out[:, i] = (y_prob[:, i] >= thr).astype(int)

    row_sums = out.sum(axis=1)
    empty_rows = np.where(row_sums == 0)[0]

    if len(empty_rows) > 0:
        max_idx = np.argmax(y_prob[empty_rows], axis=1)
        out[empty_rows, max_idx] = 1

    return out


def multilabel_rows_to_strings(y_bin: np.ndarray, labels: list) -> list:
    results = []
    for row in y_bin:
        active = [label for label, val in zip(labels, row) if val == 1]
        results.append("; ".join(active) if active else "неизвестно")
    return results


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--csv_path",
        type=str,
        default="/kaggle/input/datasets/nikitasadovoy/russian-retail/russian_retail.csv",
    )
    parser.add_argument(
        "--artifacts_dir",
        type=str,
        default="/kaggle/working/artifacts",
    )
    parser.add_argument(
        "--out_path",
        type=str,
        default="/kaggle/working/artifacts/predictions_all.csv",
    )
    args, _ = parser.parse_known_args()

    artifacts_dir = Path(args.artifacts_dir)
    out_path = Path(args.out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    print("Loading raw data...")
    df_raw = pd.read_csv(args.csv_path)

    print("Preparing features...")
    df = prepare_features(df_raw)

    result = df_raw.copy()

    # ---------- DOMAIN ----------
    domain_model_path = artifacts_dir / "domain" / "domain_model.joblib"
    domain_encoder_path = artifacts_dir / "domain" / "domain_label_encoder.joblib"

    if domain_model_path.exists():
        print("Loading domain model...")
        domain_model = joblib.load(domain_model_path)

        X_domain = sanitize_for_model(df, DOMAIN_FEATURE_COLUMNS)
        domain_pred = domain_model.predict(X_domain)

        if np.issubdtype(np.asarray(domain_pred).dtype, np.integer) and domain_encoder_path.exists():
            domain_encoder = joblib.load(domain_encoder_path)
            domain_pred = domain_encoder.inverse_transform(domain_pred)

        result["pred_domain"] = domain_pred
    else:
        print("Domain model not found, skipping.")
        result["pred_domain"] = np.nan

    # ---------- FOUNDED ----------
    founded_model_path = artifacts_dir / "founded" / "founded_model.joblib"

    if founded_model_path.exists():
        print("Loading founded model...")
        founded_model = joblib.load(founded_model_path)

        founded_df = df.copy()
        if "pred_domain" in result.columns:
            founded_df["domain"] = result["pred_domain"].fillna(founded_df.get("domain", "Неизвестно"))

        X_founded = sanitize_for_model(founded_df, FOUNDED_FEATURE_COLUMNS)
        founded_pred = founded_model.predict(X_founded)
        founded_pred = np.clip(founded_pred, 0, None)

        result["pred_founded"] = founded_pred
        result["pred_founded_rounded"] = np.round(founded_pred).astype(int)
    else:
        print("Founded model not found, skipping.")
        result["pred_founded"] = np.nan
        result["pred_founded_rounded"] = np.nan

    # ---------- PRICE CATEGORY ----------
    price_model_path = artifacts_dir / "price_category" / "price_category_model.joblib"
    price_thresholds_path = artifacts_dir / "price_category" / "price_category_thresholds.json"
    price_labels_path = artifacts_dir / "price_category" / "price_category_labels.json"

    if price_model_path.exists() and price_thresholds_path.exists() and price_labels_path.exists():
        print("Loading price_category model...")
        price_model = joblib.load(price_model_path)

        with open(price_thresholds_path, "r", encoding="utf-8") as f:
            thresholds = json.load(f)

        with open(price_labels_path, "r", encoding="utf-8") as f:
            labels = json.load(f)

        price_df = df.copy()

        if "pred_domain" in result.columns:
            price_df["domain"] = result["pred_domain"].fillna(price_df.get("domain", "Неизвестно"))

        if "pred_founded" in result.columns:
            current_founded = price_df.get("founded", pd.Series(np.nan, index=price_df.index))
            price_df["founded"] = pd.Series(result["pred_founded"], index=price_df.index).fillna(current_founded)

        X_price = sanitize_for_model(price_df, PRICE_FEATURE_COLUMNS)

        y_prob = get_price_probabilities(price_model, X_price)
        y_pred_bin = apply_thresholds(y_prob, thresholds, labels)
        y_pred_text = multilabel_rows_to_strings(y_pred_bin, labels)

        result["pred_price_category"] = y_pred_text

        for i, label in enumerate(labels):
            safe_label = (
                label.replace(" / ", "_")
                .replace(" ", "_")
                .replace("-", "_")
            )
            result[f"pred_price_prob__{safe_label}"] = y_prob[:, i]
            result[f"pred_price_label__{safe_label}"] = y_pred_bin[:, i]
    else:
        print("Price category artifacts not found, skipping.")
        result["pred_price_category"] = np.nan

    print("Saving predictions...")
    result.to_csv(out_path, index=False, encoding="utf-8-sig")

    summary = {
        "rows_scored": int(len(result)),
        "domain_model_used": bool(domain_model_path.exists()),
        "founded_model_used": bool(founded_model_path.exists()),
        "price_category_model_used": bool(price_model_path.exists()),
        "output_path": str(out_path),
    }

    with open(out_path.with_suffix(".json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("\n==============================")
    print("PREDICT ALL COMPLETE")
    print("==============================")
    print(f"Rows scored      : {len(result)}")
    print(f"Saved to         : {out_path}")


if __name__ == "__main__":
    main()

Loading raw data...
Preparing features...
Loading domain model...
Loading founded model...
Loading price_category model...
Saving predictions...

PREDICT ALL COMPLETE
Rows scored      : 2737
Saved to         : /kaggle/working/artifacts/predictions_all.csv
